# **CS Course Project — Local Differential Privacy with RAPPOR**
### *Federated Learning Simulation on the NSDUH 2021–2023 Dataset*

This project evaluates the privacy–utility tradeoff of **Local Differential Privacy (LDP)** in a
federated-learning–style simulation using the **NSDUH 2021–2023 national survey dataset**.
We simulate thousands of “clients” locally privatizing categorical records with the
**RAPPOR mechanism**, recover statistics server-side, and train decision trees on the
privatized data.

We also propose a **post-RAPPOR structural anonymization layer** that reduces linkage risk
by grouping categories via dynamic programming segmentation before learning.

---

## **Pipeline Overview**

### 1. Client-Side Privacy (Local Differential Privacy)
Each client privatizes their category using **RAPPOR**, consisting of:
- Bloom-filter encoding
- Permanent randomized response
- Instantaneous randomized response

Only noisy binary vectors are sent to the server; raw values are never shared.

---

### 2. Server-Side Statistical Decoding
The server aggregates compressed reports and **decodes category frequencies** using a
regularized linear model that inverts the Bloom + randomized-response process.
This reconstructs global statistics without identifying individuals.

---

### 3. Structural Anonymization (DP Segmentation)
We introduce a **dynamic programming segmentation step** that groups similar categories
based on:
- Decoded category frequency
- Label prevalence (label-aware segmentation)

This reduces domain resolution and improves privacy by making rare values indistinguishable.

---

### 4. Learning Models (Decision Trees)
For every drug outcome, we train three models:

**Baseline Tree**
- Trained on raw categories (no privacy)

**RAPPOR-only Tree**
- Trained on data resampled from decoded distributions

**RAPPOR + DP-seg Tree**
- Trained on segmented features

> Note: We rely on scikit-learn’s **built-in greedy split selection**.  
> A custom `best_segment_split` routine was originally... attempted but is **not used** in the final pipeline.

---

### 5. Privacy & Utility Metrics

We evaluate privacy leakage and utility using:

**Distribution Quality**
- Jensen–Shannon (JS) divergence

**Structural Leakage**
- Tree depth
- Leaf count
- JSON structure similarity

**Learning Utility**
- Accuracy
- Mutual Information (inference risk proxy)

---

### 6. Global RAPPOR Distortion Benchmark
We compute once (globally, without using labels):

- JS divergence between true and decoded distributions  
- Hamming distortion via synthetic resampling  
- Match rate between original and reconstructed categories  

---

### 7. Resolution Sweep (Granularity vs Privacy)
We repeat the full experiment across multiple domain resolutions
(e.g., K = 1500 --> 30) to study how category compression affects:

- Model accuracy  
- Statistical distortion  
- Information leakage  

We select both:
- **Best K per drug**
- **Best K per drug overall**
using a composite privacy–utility score.

---

## **Goals of the Project**

1. Implement a complete RAPPOR pipeline (encoding, decoding, evaluation)  
2. Measure how privacy affects accuracy in decision trees   
3. Quantify leakage using model structure and information measures  
4. Identify optimal resolution levels for balancing privacy and utility  

---

This notebook contains the full end-to-end implementation, evaluation, and result logging for all experiments.


## **Environment Setup & Core Utilities**

We begin by importing the Python libraries required for data processing,
model training, visualization, and experiment management.

This section also defines two global utility functions used throughout the notebook:

- **Jensen–Shannon divergence (JS)** — measures how much the RAPPOR-decoded
  distribution diverges from the true category distribution. Used as a
  privacy–utility metric.

- **normalize()** — converts noisy vectors into valid probability distributions
  after clipping or decoding. This ensures numerical stability in downstream steps.

These helpers provide the foundation for evaluating distortion, stability,
and privacy effects across the pipeline.


In [1]:
# ============================================================
# IMPORTS & HELPER FUNCTIONS
#
# This section loads all libraries used throughout the project.
# Includes:
#   - Data manipulation (numpy, pandas)
#   - Machine learning (scikit-learn)
#   - Metrics and evaluation tools
#   - Visualization utilities
#   - Model persistence (saving/loading trained trees)
# ============================================================

import numpy as np              # numerical ops and vector math
import pandas as pd             # for data processing and CSV handling
from collections import Counter # frequency counting used in distributions
import hashlib                  # hashing for RAPPOR encoding
import matplotlib.pyplot as plt # plotting utility for experiments/results
import json                     # saving metrics and model metadata
import os                       # file and directory management
import seaborn as sns
import math

# ------------------------------
# Scikit-learn components
# ------------------------------

from sklearn.tree import DecisionTreeClassifier  # ID3-style decision tree
from sklearn.model_selection import train_test_split

# compares the true labels with the model's predicted labels
from sklearn.metrics import accuracy_score       # classification accuracy metric: 

# measures the similarity between two clusterings (or label assignments) of the same data  
from sklearn.metrics import mutual_info_score    # mutual information metric  

from sklearn import tree                         # visualization and tree export
from pathlib import Path

# ------------------------------
# Tree Model imports
# ------------------------------
import joblib
from joblib import dump  # save trained model to disk
from joblib import load  # reload trained model from disk


# ------------------------------------------------------
# Utility: Jensen–Shannon Divergence
#
# Measures distribution similarity and is used as a
# privacy / utility tradeoff metric.
# ------------------------------------------------------

def js_divergence(p, q):
    '''
    Computes Jensen–Shannon divergence between two probability distributions.
    Quantifies how much information changes between the original
    and privatized distributions.

    JS is:
        - symmetric
        - always finite
        - bounded between 0 and 1 (if using log base 2), which we do

    Parameters:
        p (array-like):
            Probability distribution from original data.
        q (array-like):
            Probability distribution from privatized data.

    Return value(s):
        js (float):
            Jensen–Shannon divergence in bits.
    '''
    
    # convert to numpy arrays for safe math operations
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    # force valid probability distributions
    p = p / p.sum()
    q = q / q.sum()

    # midpoint distribution
    m = 0.5 * (p + q)

    def kl(a, b):
        # avoid log(0) by ignoring zero entries
        mask = (a > 0)
        return np.sum(a[mask] * np.log2(a[mask] / b[mask]))

    # JS divergence is average of two KL divergences
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


# ------------------------------------------------------
# Utility: Normalize noisy vectors
#
# Ensures privatized outputs remain valid
# probability distributions after noise injection.
# ------------------------------------------------------

def normalize(v):
    '''
    Converts a noisy vector into a valid probability distribution.
    Used after differential privacy noise is introduced.

    This prevents:
        - negative values
        - division by zero
        - invalid probability vectors

    Parameters:
        v (array-like):
            Noisy count or frequency vector.

    Return value(s):
        v_norm (array-like):
            Valid probability distribution that sums to 1.
    '''
    
    # clip negative values caused by DP noise
    v = np.maximum(v, 0)

    # normalize safely
    s = v.sum()
    return v / s if s > 0 else np.ones_like(v) / len(v)


# check check checking....
print("Imports loaded successfully.")

Imports loaded successfully.


## **Dataset Loading (NSDUH 2021–2023)**

We load the merged NSDUH 2021–2023 public-use dataset (tab-separated format).
The dataset contains approximately **173,000 respondents** and thousands of
survey variables covering:

- demographics  
- education and income  
- psychological distress  
- substance use  
- health behaviors  

Official NSDUH missing-value codes (`-9, -8, -7`) are converted to `NaN` for
consistent processing.

Each row represents a single simulated **federated client**, whose data will
later be privatized locally using RAPPOR before aggregation.

In [2]:
# ============================================================
# LOAD NSDUH 2021–2023 DATASET
#
# Reads in the National Survey on Drug Use and Health data
# containing anonymous survey responses used as the simulated
# population for federated learning clients.
#
# This is the "RAW" dataset before any:
#   - anonymization
#   - RAPPOR encoding
#   - compression
#   - differential privacy
#
# All privacy protections are applied AFTER this stage.
# ============================================================

DATA_PATH = "NSDUH_2021_2023_Tab.txt"   # file must(!) exist in same directory

df = pd.read_csv(
    DATA_PATH,
    sep="\t",               # NSDUH files are tab-delimited
    low_memory=False,       # avoids pandas type guessing warnings on large files
    na_values=[-9, -8, -7]  # official NSDUH missing-value codes
)

# check check checking....
print("Dataset successfully loaded!")
print("Shape (rows, columns):", df.shape)

# tiny check 
# df.head()

Dataset successfully loaded!
Shape (rows, columns): (173808, 2639)


## **Feature Engineering: High-Dimensional Categorical Encoding**

RAPPOR operates on **categorical values**, so we transform multiple NSDUH attributes
into a single high-cardinality categorical feature.

For each respondent, we construct a tuple using:

- Age group (`CATAG3`)  
- Sex (`IRSEX`)  
- Race (`NEWRACE2`)  
- Education level (`EDUHIGHCAT`)  
- Poverty category (`POVERTY3`)  
- K6 psychological distress score (binned into 3 levels)  

Each row maps to a **6-dimensional categorical profile**, which is then assigned
a unique integer ID. These IDs represent the true client-side values that are
later privatized using RAPPOR.

Rows with missing values in any required field are filtered using a binary mask
to ensure consistent feature encoding.


In [3]:
def make_features(df):
    '''
    Constructs categorical feature representation
    (~1600 distinct combinations) from demographic and mental-health
    attributes.

    Each row is mapped to a single categorical ID representing:
        (age_bin, sex, race, education, poverty, distress_bin)

    This encoding forms the BASE representation used before:
        - RAPPOR encoding
        - category compression
        - DP segmentation

    Parameters:
        df (DataFrame):
            Raw NSDUH dataset as loaded from disk.

    Return value(s):
        X_features (np.array):
            Encoded feature IDs per row (categories).
        mapping_features (dict):
            Maps each feature tuple to a unique integer ID.
        mask_features (np.array):
            Boolean mask identifying rows with valid data.
    '''

    # -----------------------
    # Gather in the base attributes
    # -----------------------

    age        = df["CATAG3"]         # age group
    sex        = df["IRSEX"]          # biological sex
    race       = df["NEWRACE2"]       # race category
    education  = df["EDUHIGHCAT"]     # highest education level
    poverty    = df["POVERTY3"]       # income-to-poverty ratio

    # -----------------------
    # K6 psychological distress indicators
    # -----------------------

    k6_cols = [
        "DSTRST30", "DSTHOP30", "DSTNRV30",
        "DSTCHR30", "DSTEFF30", "DSTNGD30"
    ]
    k6 = df[k6_cols]

    # -----------------------
    # Construct validity mask
    #
    # Ensures only rows with complete information
    # are used in the experiment pipeline.
    # -----------------------

    base_valid = (
        age.notna() &
        sex.notna() &
        race.notna() &
        education.notna() &
        poverty.notna() &
        k6.notna().all(axis=1)
    )

    mask_features = base_valid.to_numpy()

    # -----------------------
    # Filter invalid rows
    # -----------------------

    df_valid = df.loc[mask_features]

    # -----------------------
    # Compute K6 distress score
    #
    # Higher scores indicate greater psychological distress.
    # -----------------------

    k6_score = df_valid[k6_cols].fillna(0).sum(axis=1)

    # -----------------------
    # Discretize distress into 3 bins
    #
    # 0 = low distress
    # 1 = moderate distress
    # 2 = high distress
    # -----------------------

    distress_bin = pd.cut(
        k6_score,
        bins=[-1, 4, 12, 100],
        labels=[0, 1, 2]
    )

    # pd.cut may produce NaN for edge cases
    distress_bin = distress_bin.astype("float").fillna(0).astype(int)

    # -----------------------
    # Convert categorical values into integer bins
    #
    # These are assumed to be pre-coded by NSDUH.
    # -----------------------

    age_bin   = df_valid["CATAG3"].astype(int)
    sex_bin   = df_valid["IRSEX"].astype(int)
    race_bin  = df_valid["NEWRACE2"].astype(int)
    edu_bin   = df_valid["EDUHIGHCAT"].astype(int)
    pov_bin   = df_valid["POVERTY3"].astype(int)

    # -----------------------
    # Combine into categorical feature tuples
    #
    # Each tuple = one "demographic profile"
    # -----------------------

    tuples = list(zip(
        age_bin,
        sex_bin,
        race_bin,
        edu_bin,
        pov_bin,
        distress_bin
    ))

    # -----------------------
    # Assign unique category IDs
    #
    # Each distinct combination becomes one class ID.
    # -----------------------

    mapping_features = {}
    ids = []
    next_id = 0

    for t in tuples:
        if t not in mapping_features:
            mapping_features[t] = next_id
            next_id += 1
        ids.append(mapping_features[t])

    X_features = np.array(ids, dtype=int)

    return X_features, mapping_features, mask_features


In [4]:
# ------------------------------------------------------------
# Apply feature construction and filter dataset
#
# Runs feature encoding and removes invalid rows based on
# missing demographic or distress information.
# ------------------------------------------------------------

X_features, mapping_features, mask_features = make_features(df)

# keep only valid rows after feature masking
df_filtered = df[mask_features].reset_index(drop=True)
X_filtered  = X_features

# confirm outputs and making sure, everything looks good
# a check check checking....
print("After feature-mask filtering:")
print(" df_filtered shape:", df_filtered.shape)
print(" X_filtered shape:", X_filtered.shape)
print(" Number of unique categories:", len(mapping_features))

After feature-mask filtering:
 df_filtered shape: (173729, 2639)
 X_filtered shape: (173729,)
 Number of unique categories: 1638


## **Drug Label Construction (Binary Targets)**

For supervised learning, NSDUH drug-use variables are converted into **binary
classification labels** indicating use or misuse.

Label definitions follow survey conventions:

- **Illicit drugs** (e.g., cocaine, heroin, methamphetamine, hallucinogens)  
  → labeled `1` if usage is reported at least once in the past year.

- **Tobacco and nicotine vaping**  
  → use NSDUH’s binary coding (`0 = no`, `1 = yes`).

- **Prescription misuse categories**  
  → labeled `1` if misuse is reported.

Invalid survey responses (`85, 94, 97, 98, 99`) are filtered using a mask.

For each drug, this produces:

- `y` — binary label vector  
- `mask_y` — validity mask indicating usable rows  
- `X_cat` — category features aligned with valid labels  

Each drug outcome is trained and evaluated independently using a decision tree.


In [5]:
##############################################################
# DRUG LABEL CONSTRUCTION (NSDUH -> binary y)
#
# Converts survey drug-use variables into binary classification
# labels indicating presence or absence of use/misuse.
#
# This defines the PREDICTION TARGET for all tree models.
# Privacy risk begins here because behavior is introduced.
##############################################################

# Mapping from experiment drug names -> NSDUH variable codes
DRUG_VARIABLES = {
    # Illicit drug use
    "marijuana": "MRJYR",
    "cocaine": "COCYR",
    "crack": "CRKYR",
    "heroin": "HERYR",
    "methamphetamine": "METHAMYR",
    "inhalants": "INHALYR",
    "hallucinogens": "HALLUCYR",

    # Tobacco
    "tobacco": "CIGYR",

    # Nicotine vaping
    "nicotine_vaping": "NICVAPYR", # only introduce after 2021 year

    # Prescription misuse
    "opioid_misuse": "OPIANYYR",
    "tranquilizer_misuse": "TRQANYYR",
    "sedative_misuse": "SEDANYYR",
    "stimulant_misuse": "STMANYYR",
    "pain_reliever_misuse": "PNRANYYR"
}

# ------------------------------------------------------------
# Drug groups for "global" trees (one outcome per drug type)
# ------------------------------------------------------------
DRUG_GROUPS = {
    "illicit": [
        "marijuana", "cocaine", "crack", "heroin",
        "methamphetamine", "inhalants", "hallucinogens"
    ],
    "nicotine": [
        "tobacco", "nicotine_vaping"
    ],
    "prescription_misuse": [
        "opioid_misuse", "tranquilizer_misuse",
        "sedative_misuse", "stimulant_misuse",
        "pain_reliever_misuse"
    ]
}

# NSDUH invalid / non-response codes
INVALID_CODES = {85, 94, 97, 98, 99}


def make_drug_labels(df, drug_name):
    '''
    Converts NSDUH drug-use variables into binary labels.

    Each participant is assigned:
        0 -> no reported use
        1 -> use or misuse reported

    Invalid survey responses are filtered using a mask.

    Parameters:
        df (DataFrame):
            Filtered NSDUH dataset.
        drug_name (str):
            One of the keys in DRUG_VARIABLES.

    Return value(s):
        y (np.array):
            Binary label vector.
        mask_y (np.array):
            Boolean mask identifying valid responses.
    '''

    # verify label exists
    if drug_name not in DRUG_VARIABLES:
        raise ValueError(f"Unknown drug '{drug_name}'")

    col = DRUG_VARIABLES[drug_name]

    raw = df[col].reset_index(drop=True)

    # identify valid survey responses
    mask_y = (~raw.isin(INVALID_CODES) & raw.notna()).to_numpy()

    clean = raw[mask_y]

    # ---------------------------------
    # Usage logic by survey format type
    # (a cleaning if you will)
    # ---------------------------------

    # marijuana and illicit drugs: record number of days used
    if drug_name in [
        "marijuana", "cocaine", "crack", "heroin",
        "methamphetamine", "inhalants", "hallucinogens"
    ]:
        y = (clean > 0).astype(int)

    # tobacco and vaping are binary flags
    elif drug_name in ["tobacco", "nicotine_vaping"]:
        y = (clean == 1).astype(int)

    # prescription misuse already encoded as binary
    elif drug_name in [
        "opioid_misuse", "tranquilizer_misuse",
        "sedative_misuse", "stimulant_misuse",
        "pain_reliever_misuse"
    ]:
        y = (clean == 1).astype(int)

    else:
        # safety fallback
        y = (clean > 0).astype(int)

    return y.to_numpy(), mask_y

def make_group_labels(df, group_name, member_drugs):
    """
    Builds a combined binary label for a drug group.

    - Only respondents with valid data for *all* member drugs are kept.
    - Group label = 1 if ANY member drug has label 1, else 0.

    Returns:
        y_group (np.array) : binary labels
        mask_group (np.array, bool) : mask applied to df_filtered / X_filtered
    """

    n_rows = df.shape[0]
    # start with "everyone included", then intersect validity
    valid_mask = np.ones(n_rows, dtype=bool)

    # store full-length aligned labels for each member drug
    aligned_labels = []

    for drug in member_drugs:
        y_raw, mask_y = make_drug_labels(df, drug)

        # expand y_raw back to full length (aligned with df rows)
        full = np.full(n_rows, np.nan)
        full[mask_y] = y_raw

        # valid where we have a non-NaN label
        valid_mask &= ~np.isnan(full)
        aligned_labels.append(full)

    # now compute group label on the intersection mask
    group_y = np.zeros(n_rows, dtype=int)
    for full in aligned_labels:
        group_y[valid_mask] |= (full[valid_mask] == 1).astype(int)

    # compress to only valid rows, like make_drug_labels does
    y_group = group_y[valid_mask]

    return y_group, valid_mask

# --------------------------
# Example: build labels
# using same test durg throughout
# --------------------------

drug_test = "marijuana"

y_raw, mask_y = make_drug_labels(df_filtered, drug_test)

# align features with valid labels
X_final = X_filtered[mask_y]
y_final = y_raw

## **RAPPOR Client-Side Encoding (Local Differential Privacy)**

This section implements the full **RAPPOR mechanism**, originally developed by
Google.

Each simulated client applies RAPPOR locally before sending any data:

### Encoding pipeline

1. **Bloom filter encoding**  
   Maps each category to a fixed-length binary vector using multiple hash functions.

2. **Permanent Randomized Response**  
   Introduces client-level randomness, preventing long-term tracking or
   repeated-identification attacks.

3. **Instantaneous Randomized Response**  
   Adds fresh randomness per report, enforcing strong local privacy guarantees.

The function `rappor_client_reports()` simulates one locally privatized report
per user, enforcing **Local Differential Privacy** before any aggregation occurs.


In [6]:
##############################################################
# RAPPOR PARAMETERs SETUP!!
#
# These values define the privacy and accuracy behavior
# of the RAPPOR encoding process.
#
# Larger noise -> stronger privacy, lower utility
# Smaller noise -> weaker privacy, higher utility
##############################################################

def rappor_epsilon(p, q):
    """
    Compute the local differential privacy (epsilon) of RAPPOR
    from instantaneous randomized response parameters.

    ε = ln( q(1 - p) / ( p(1 - q) ) )
    """
    return math.log((q * (1 - p)) / (p * (1 - q)))

# RAPPOR parameters
BLOOM_K = 128     # bits in Bloom filter
BLOOM_H = 2       # number of hash functions
F       = 0.5     # permanent RR flip prob
P       = 0.5     # instantaneous RR prob when base bit = 0
Q       = 0.75    # instantaneous RR prob when base bit = 1

EPSILON = rappor_epsilon(P, Q)
print(f"RAPPOR Local Privacy Budget ε = {EPSILON:.4f}") # from these parameters (Q and P), ε ≈ 1.10


RAPPOR Local Privacy Budget ε = 1.0986


In [7]:
##############################################################
# RAPPOR CLIENT-SIDE ENCODING
#
# Implements the RAPPOR mechanism:
#   1) Hash category -> Bloom filter
#   2) Apply permanent randomized response
#   3) Apply instantaneous randomized response
#   4) Generate client-side privacy-preserving reports
#
# This enforces LOCAL DIFFERENTIAL PRIVACY. (LDP!!!) (ง •̀_•́)ง
##############################################################

def hash_indices(value_id, k=BLOOM_K, h=BLOOM_H, salt=""):
    '''
    Maps a categorical value ID to Bloom filter positions.

    Uses SHA-256 hashing to simulate multiple independent hash functions.

    Parameters:
        value_id (int):
            Category ID to encode.
        k (int):
            Bloom filter size (number of bits).
        h (int):
            Number of hash functions.
        salt (str):
            Optional salt to prevent cross-domain linkage.

    Return value(s):
        indices (list):
            Bloom filter indices to activate.
    '''

    indices = []

    # hash category value and salt
    base = f"{value_id}:{salt}".encode("utf-8")
    digest = hashlib.sha256(base).digest()  # 256-bit hash

    # carve hash stream into 4-byte chunks → indices
    for i in range(h):
        chunk = digest[4*i:4*(i+1)]
        idx = int.from_bytes(chunk, byteorder="big") % k
        indices.append(idx)

    return indices


def bloom_encode_category(cat_id, k=BLOOM_K, h=BLOOM_H):
    '''
    Encodes a category ID as a Bloom filter bit vector.

    Parameters:
        cat_id (int):
            Category identifier.
        k (int):
            Bloom filter length.
        h (int):
            Number of hashes.

    Return value(s):
        bits (np.array):
            Binary Bloom filter vector.
    '''
    
    bits = np.zeros(k, dtype=int)

    # set hash-selected indices to 1
    for idx in hash_indices(cat_id, k=k, h=h, salt="rappor"):
        bits[idx] = 1

    return bits


def permanent_rr(bloom_bits, f=F, rng=None):
    '''
    Applies permanent randomized response to Bloom bits.

    Each bit is:
        - replaced with random noise with probability f
        - preserved with probability (1 - f)

    This makes each client representation stable across reports
    but permanently obfuscated.

    Parameters:
        bloom_bits (array-like):
            Clean Bloom vector.
        f (float):
            Noise probability.
        rng (Generator):
            Numpy random generator.

    Return value(s):
        perm_bits (np.array):
            Permanently perturbed Bloom vector.
    '''

    if rng is None:
        rng = np.random.default_rng()

    # decide which bits to randomize
    flip_mask = rng.random(size=bloom_bits.shape) < f

    # generate replacement randomness
    random_bits = rng.integers(0, 2, size=bloom_bits.shape)

    perm_bits = bloom_bits.copy()
    perm_bits[flip_mask] = random_bits[flip_mask]

    return perm_bits


def instantaneous_rr(perm_bits, p=P, q=Q, rng=None):
    '''
    Applies real-time randomized response.

        If bit = 1 -> return 1 with probability q
        If bit = 0 -> return 1 with probability p

    Parameters:
        perm_bits (array-like):
            Permanently randomized Bloom vector.
        p (float):
            False positive probability.
        q (float):
            True positive probability.

    Return value(s):
        out (np.array):
            Final privatized bit vector.
    '''

    if rng is None:
        rng = np.random.default_rng()

    r = rng.random(size=perm_bits.shape)

    out = np.zeros_like(perm_bits)

    ones  = (perm_bits == 1)
    zeros = ~ones

    # prob bit reporting
    out[ones]  = (r[ones] < q).astype(int)
    out[zeros] = (r[zeros] < p).astype(int)

    return out


def rappor_client_reports(X_cat, k=BLOOM_K, h=BLOOM_H, f=F, p=P, q=Q, seed=None):
    '''
    Simulates client-side RAPPOR encoding for all users.

    Each value is:
        category -> bloom -> permanent noise -> instantaneous noise

    Parameters:
        X_cat (array-like):
            Category ID per user.
        k, h, f, p, q:
            RAPPOR parameters.
        seed (int):
            RNG seed for reproducibility.

    Return value(s):
        reports (np.array):
            Binary RAPPOR reports, shape (n_users, bloom_size).
    '''

    rng = np.random.default_rng(seed)
    n   = len(X_cat)

    reports = np.zeros((n, k), dtype=int)

    for i, cat_id in enumerate(X_cat):
        bloom = bloom_encode_category(cat_id, k=k, h=h)
        perm  = permanent_rr(bloom, f=f, rng=rng)
        inst  = instantaneous_rr(perm, p=p, q=q, rng=rng)

        reports[i, :] = inst

    return reports

## **Server-Side Aggregation & Decoding**

After receiving locally privatized reports from clients, the server:

- aggregates Bloom-filter bits across all users  
- constructs a **design matrix** modeling expected noisy behavior for each category  
- solves a **ridge-regularized linear system** to recover category frequencies  

This statistical decoding reconstructs global patterns without uncovering any
individual record.

The output is:

- `est_counts` — estimated number of users per category  
- `est_probs` — estimated category probability distribution  

These decoded distributions represent the server’s best reconstruction of the categorical feature under LDP noise and are later used to train models.

In [8]:
##############################################################
# RAPPOR SERVER-SIDE AGGREGATION & DECODING!!!
#
# This section:
#   - aggregates client RAPPOR reports
#   - reconstructs estimated frequency distributions
#   - decodes Bloom-filter noise using linear inversion
#
# This is where noise is removed statistically WITHOUT ((ง •̀_•́)ง!!!)
# revealing any individual's original data.
##############################################################

def rappor_aggregate(reports):
    '''
    Aggregates RAPPOR client reports by summing bits position-wise.

    Parameters:
        reports (np.array):
            Matrix of RAPPOR reports, shape (n_users, bloom_size).

    Return value(s):
        counts (np.array):
            Vector of bit totals, length bloom_size.
        n (int):
            Number of users.
    '''

    n = reports.shape[0]
    counts = reports.sum(axis=0)
    return counts, n


def build_rappor_design_matrix(K, k=BLOOM_K, h=BLOOM_H, f=F, p=P, q=Q):
    '''
    Constructs the expected response probability matrix A where:

        A[i, j] = P(final_bit[i] = 1 | category j)

    This matrix captures:
        - Bloom filter assignment
        - permanent randomized response
        - instantaneous randomized response

    Parameters:
        K (int):
            Number of categories.
        k (int):
            Bloom size.
        h (int):
            Number of hash functions.
        f, p, q:
            RAPPOR parameters.

    Return value(s):
        A_prob (np.array):
            Expected probability matrix with shape (bloom_size, num_categories).
    '''

    # compute bloom patterns for every category
    bloom_bits = np.zeros((K, k), dtype=int)
    for j in range(K):
        bloom_bits[j, :] = bloom_encode_category(j, k=k, h=h)

    # permanent randomized response
    alpha1 = 1.0 - f / 2.0     # P(perm = 1 | orig = 1)
    beta1  = f / 2.0           # P(perm = 1 | orig = 0)

    # instantaneous randomized response
    prob1 = alpha1 * q + (1.0 - alpha1) * p   # P(final=1 | orig=1)
    prob0 = beta1  * q + (1.0 - beta1)  * p   # P(final=1 | orig=0)

    # build expected response matrix
    A_prob = np.zeros((k, K), dtype=float)
    for j in range(K):
        for i in range(k):
            A_prob[i, j] = prob1 if bloom_bits[j, i] == 1 else prob0

    return A_prob


def rappor_decode(counts, n, K, k=BLOOM_K, h=BLOOM_H, f=F, p=P, q=Q, l2_reg=1e-2):
    '''
    Estimates category frequencies from aggregated RAPPOR reports.

    Uses ridge-regularized least squares:

        minimize ||A x - y||^2 + l2_reg ||x||^2

    Enforces (ง •̀_•́)ง:
        - non-negativity
        - distribution normalization

    Parameters:
        counts (np.array):
            Sum of reported Bloom bits.
        n (int):
            Total number of users.
        K (int):
            Number of original categories.
        l2_reg (float):
            Regularization strength.

    Return value(s):
        est_counts (np.array):
            Estimated number of users per category.
        est_probs (np.array):
            Estimated probability distribution.
    '''

    # observed bit probabilities
    y_prob = counts / float(n)

    # expected probabilities from encoding model
    A = build_rappor_design_matrix(K, k=k, h=h, f=f, p=p, q=q)

    
    # -----------------------------
    # Solve linear system via normal equations
    #
    # We solve the ridge regression system:
    #
    #   (A.T A + l2_reg*I) x = A.T y
    #
    # where:
    #   A.T = transpose of A
    #   @  = matrix multiplication
    #   I  = identity matrix
    #   l2_reg = lambda -> meant to stablize 
    #            noisy matrix inversion
    # -----------------------------
    
    ATA = A.T @ A                       # A.T A -> feature correlation matrix    
    ATy = A.T @ y_prob                  # A.T y -> projected observations
    ATA_reg = ATA + l2_reg * np.eye(K)  # l2_reg * I -> ridge regularization

    # solve linear system: (A.T A + l2_reg * I)x = A.T y
    x_hat = np.linalg.solve(ATA_reg, ATy)

    
    # -----------------------------
    # Enforce (ง •̀_•́)ง probability constraints
    # -----------------------------

    # remove negative estimates caused by noise
    x_hat = np.clip(x_hat, 0, None)

    # normalize into valid probability distribution
    if x_hat.sum() > 0:
        x_hat = x_hat / x_hat.sum()
    else:
        x_hat = np.ones(K) / K    # fallback: uniform

    # yay yay yay
    est_probs = x_hat
    est_counts = est_probs * n

    return est_counts, est_probs


In [9]:
# ------------------------------------------------------------
# Execute RAPPOR pipeline and recover estimated distribution
# ------------------------------------------------------------

K = len(mapping_features) # num of og categories

# distribution from "raw" data
true_counts = np.bincount(X_final, minlength=K)
true_probs  = true_counts / true_counts.sum()

# run the RAPPOR (reports, agg., decode)
reports = rappor_client_reports(X_final, seed=42)
counts, n = rappor_aggregate(reports)
est_counts, est_probs = rappor_decode(counts, n, K)

# normalize just to be supa safe
est_probs = np.clip(est_probs, 0, None)
est_probs = est_probs / est_probs.sum()

## **Proposed Algorithm: Label-Aware DP Segmentation**

RAPPOR introduces substantial noise, especially in high-cardinality domains.
To mitigate this, we introduce a **post-processing anonymization layer** that
structurally compresses the domain *after decoding*, without weakening the
local-DP guarantee.

### 1. Label-Aware Scoring  
For each category, we compute a combined score using:

- RAPPOR-estimated frequency  
- category-level label prevalence  

This prioritizes categories that are both common and predictive.

---

### 2. Dynamic Programming Segmentation  
Categories are sorted by their combined score and segmented into **B contiguous
groups** using dynamic programming to minimize within-segment variance.

This creates smoother, lower-resolution features that reduce sparsity and
linkability.

---

### 3. Decision Tree Learning on Segments  
The resulting segments replace the original categories as model features.
A standard decision tree is trained using scikit-learn’s built-in splitter.

No custom split rule is used — segmentation is the only structural modification.

---

This approach improves privacy by reducing domain granularity while preserving
learning utility, and remains compatible with Local Differential Privacy because
it operates entirely *after* privatization.


In [10]:
##############################################################
# DP-BASED SEGMENTATION (CATEGORY MERGING!!!)
#
# Clusters categories into B contiguous segments based on their
# estimated frequency distribution.
#
# This reduces:
#   - domain granularity
#   - linkage risk
#   - sparsity
#   - re-identification exposure
#
# It trades resolution for privacy.
##############################################################


def dp_segment_by_values(values, B):
    '''
    Partitions categories into B contiguous segments using
    dynamic programming to minimize within-segment variance.

    This groups similar-frequency categories together to reduce
    distinguishability between sparse values.

    Parameters:
        values (np.array):
            Estimated probabilities per category (length K).
        B (int):
            Desired number of segments.

    Return value(s):
        segment_id (np.array):
            Maps each category -> assigned segment.
        segments (list):
            Lists of category indices per segment.
    '''

    K = len(values)

    # sort categories by frequency
    order = np.argsort(values)
    v = values[order]

    # --------------------------------------------------
    # Precompute prefix sums for O(1) variance queries
    # --------------------------------------------------

    prefix_sum   = np.zeros(K + 1)
    prefix_sumsq = np.zeros(K + 1)

    for i in range(1, K + 1):
        prefix_sum[i]   = prefix_sum[i-1]   + v[i-1]
        prefix_sumsq[i] = prefix_sumsq[i-1] + v[i-1]**2

    def seg_cost(i, j):
        '''
        Computes sum of squared error of a segment [i..j].

        Used as the objective measure for segmentation quality:
        low variance -> more homogeneous (same) categories.
        '''
        length = j - i + 1
        s  = prefix_sum[j+1]   - prefix_sum[i]
        ss = prefix_sumsq[j+1] - prefix_sumsq[i]

        mean = s / length

        # SSE = the sum of[ (v - mean)^2 ]
        return ss - 2*mean*s + length*(mean**2)

    # --------------------------------------------------
    # Dynamic Programming Setup
    #
    # dp[b][j] = minimum segmentation cost of first j values
    #            into b segments
    # --------------------------------------------------

    INF = float("inf")
    dp   = np.full((B + 1, K + 1), INF)
    prev = np.full((B + 1, K + 1), -1, dtype=int)

    # base case: single segment covering [0..j-1]
    for j in range(1, K + 1):
        dp[1][j] = seg_cost(0, j-1)
        prev[1][j] = 0

    # DP recurrence
    for b in range(2, B + 1):
        for j in range(b, K + 1):
            best_cost = INF
            best_i = -1

            # try all possible previous splits
            for i in range(b-1, j):
                cost = dp[b-1][i] + seg_cost(i, j-1)
                if cost < best_cost:
                    best_cost = cost
                    best_i = i

            dp[b][j] = best_cost
            prev[b][j] = best_i

    # --------------------------------------------------
    # Recover segment boundaries via backtracking
    # --------------------------------------------------

    boundaries = []
    b = B
    j = K

    while b > 0:
        i = prev[b][j]
        boundaries.append((i, j-1))
        j = i
        b -= 1

    boundaries.reverse()

    # --------------------------------------------------
    # Assign segment IDs
    # --------------------------------------------------

    segment_id = np.zeros(K, dtype=int)
    segments = []

    for seg_idx, (start, end) in enumerate(boundaries):
        seg_positions = np.arange(start, end+1)
        seg_cats = order[seg_positions]

        segments.append(seg_cats)

        for c in seg_cats:
            segment_id[c] = seg_idx

    return segment_id, segments


In [11]:
"""
def best_segment_split(seg_total, seg_pos):
    '''
    Finds the optimal binary split over ordered segments using
    classification accuracy as the objective function. (greedy...) muahahaha

    Only contiguous prefix splits are considered:
        [0..t-1] | [t..B-1]

    Parameters:
        seg_total (np.array):
            Total number of samples per segment.
        seg_pos (np.array):
            Number of positive-class samples per segment.

    Return value(s):
        best_left (int):
            Segment index where split occurs.
        best_acc (float):
            Maximum achievable accuracy from the split.
    '''

    B = len(seg_total)

    total_all = seg_total.sum()
    pos_all   = seg_pos.sum()
    neg_all   = total_all - pos_all

    best_acc = 0
    best_left = None

    # CUMULATIVE counts for prefix splits
    cumulat_total = np.cumsum(seg_total)
    cumulat_pos   = np.cumsum(seg_pos)

    # evaluate every contiguous split
    for t in range(1, B):

        left_total = cumulat_total[t-1]
        left_pos   = cumulat_pos[t-1]
        left_neg   = left_total - left_pos

        right_total = total_all - left_total
        right_pos   = pos_all - left_pos
        right_neg   = right_total - right_pos

        # skip degenerate (•̀⤙•́ ) splits
        if left_total == 0 or right_total == 0:
            continue

        # majority vote on each side
        left_correct  = max(left_pos, left_neg)
        right_correct = max(right_pos, right_neg)

        acc = (left_correct + right_correct) / total_all

        # select best split
        if acc > best_acc:
            best_acc = acc
            best_left = t

    return best_left, best_acc
"""
omg = 'wow I dont need this actually at all.. which is kinda sad I feel but that is okay'

## **Modeling & Evaluation Framework**

From this point forward, the notebook executes the full experimental pipeline
for evaluating privacy, utility, and structural leakage.

For each drug outcome, we compare three learning pipelines:

1. **Baseline Tree**  
   Trained on raw categorical features (no privacy).

2. **RAPPOR-Only Tree**  
   Trained using categories sampled from decoded RAPPOR distributions.

3. **RAPPOR + DP-Segmented Tree**  
   Trained on structurally anonymized features produced by DP segmentation.

Beyond classification accuracy (true labels vs predicted), we assess:

- distribution distortion (JS divergence)  
- structural privacy (tree depth, leaf count, tree similarity)  
- information leakage (mutual information)  

We also perform a resolution sweep to evaluate how category granularity
affects privacy–utility tradeoffs.

This framework simulates a federated setting in which the server never accesses
raw client values and learning occurs entirely on locally privatized data.


### Helper functions for our pipelines

In [12]:
##############################################################
# LABEL-AWARE DP SEGMENTATION & TREE UTILITIES
#
# This section helps with:
#   - RAPPOR-decoded distributions
#   - label statistics
#   - segmentation logic
#   - simple decision-tree training
##############################################################

def category_label_counts(X, y, K):
    '''
    Computes per-category counts and label frequencies.

    Parameters:
        X (array-like):
            Category ID per sample.
        y (array-like):
            Binary label (0/1).
        K (int):
            Number of categories.

    Return value(s):
        cat_total (np.array):
            Samples per category.
        cat_pos (np.array):
            Positive labels per category.
    '''

    cat_total = np.zeros(K, dtype=int)
    cat_pos   = np.zeros(K, dtype=int)

    for xi, yi in zip(X, y):
        cat_total[xi] += 1
        if yi == 1:
            cat_pos[xi] += 1

    return cat_total, cat_pos


def label_aware_scores(est_probs, cat_rate, alpha=0.7):
    '''
    Combines:
        - estimated category frequency
        - label prevalence

    Produces a single score used for segmentation ordering.

    Parameters:
        est_probs (array-like):
            RAPPOR-decoded distribution.
        cat_rate (array-like):
            Positive label fraction per category.
        alpha (float):
            Weight assigned to label influence.

    Return value(s):
        scores (np.array):
            Combined category score.
    '''

    est_probs = np.asarray(est_probs, dtype=float)
    cat_rate  = np.asarray(cat_rate, dtype=float)

    # normalize decoded probability vector
    if est_probs.sum() > 0:
        p_norm = est_probs / est_probs.sum()
    else:
        p_norm = np.ones_like(est_probs) / len(est_probs)

    # normalize label prevalence into 0..1 range
    r_min = cat_rate.min()
    r_max = cat_rate.max()
    if r_max > r_min:
        r_norm = (cat_rate - r_min) / (r_max - r_min)
    else:
        r_norm = np.zeros_like(cat_rate)

    # blend rate and frequency
    scores = alpha * r_norm + (1.0 - alpha) * p_norm

    return scores


def dp_segment_label_aware(est_probs, cat_rate, B=30, alpha=0.7):
    '''
    Performs segmentation using both:
        - privacy-preserved frequencies
        - class skew information

    Parameters:
        est_probs (np.array):
            RAPPOR-estimated probabilities.
        cat_rate (np.array):
            Positive-class ratios.
        B (int):
            Number of target segments.
        alpha (float):
            Label weighting strength.

    Return value(s):
        segment_id (np.array):
            Category-to-segment mapping.
        segments (list):
            Grouped category indices.
    '''

    scores = label_aware_scores(est_probs, cat_rate, alpha=alpha)

    # cluster categories by combined utility score
    segment_id, segments = dp_segment_by_values(scores, B=B)

    return segment_id, segments


def train_eval_tree_1d(X, y, max_depth=4, random_state=0):
    '''
    Trains a simple decision tree using one categorical input.

    Parameters:
        X (array-like):
            Single feature vector.
        y (array-like):
            Binary labels.
        max_depth (int):
            Tree depth.
        random_state (int):
            RNG seed.

    Return value(s):
        acc (float):
            Training accuracy.
        clf (DecisionTreeClassifier):
            Fitted tree.
    '''

    X = X.reshape(-1,1)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=random_state, stratify=y
    )

    clf = DecisionTreeClassifier(max_depth=max_depth, random_state=random_state)
    clf.fit(X_train, y_train)

    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)

    return acc, clf


In [13]:
##############################################################
# MODEL EXPORT, REPORTING, & STRUCTURAL PRIVACY ANALYSIS
#
# Utilities for:
#   - turning trained trees to JSON
#   - writing experiment summaries (a nice little readme :)
#   - quantifying structural leakage via tree complexity
# ##############################################################

def sklearn_tree_to_json(decision_tree):
    '''
    Converts a scikit-learn DecisionTree model into
    a recursive JSON representation.

    This allows:
        - visualization
        - leakage analysis

    Parameters:
        decision_tree (DecisionTreeClassifier):
            Trained model.

    Return value(s):
        tree_json (dict):
            Recursive tree structure as JSON.
    '''

    tree_ = decision_tree.tree_

    def recurse(node):

        # leaf node
        if tree_.feature[node] == -2:
            return {
                "type": "leaf",
                "samples": int(tree_.n_node_samples[node]),
                "value": tree_.value[node].tolist()
            }

        # internal split
        return {
            "type": "node",
            "feature": int(tree_.feature[node]),
            "threshold": float(tree_.threshold[node]),
            "samples": int(tree_.n_node_samples[node]),
            "left": recurse(tree_.children_left[node]),
            "right": recurse(tree_.children_right[node])
        }

    return recurse(0)


def write_experiment_readme(drug_dir, meta, metrics):
    '''
    Writes a wittle summary of experiment results.

    Records:
        - dataset configuration
        - segmentation parameters
        - model performance
        - structural privacy metrics
        - mutual information (inference risk proxy)
    '''

    readme = f"""
EXPERIMENT SUMMARY: {meta.get('drug', meta.get('group_name', '')).upper()}

===========================================
DATASET
-------------------------------------------
Total samples: {meta['n']}
Positive prevalence: {meta['prevalence']:.4f}

===========================================
MODEL CONFIGURATION
-------------------------------------------
Max depth: {meta['max_depth']}
DP segments (B): {meta['B']}
Segment smoothing alpha: {meta['alpha']}

===========================================
MODEL PERFORMANCE
-------------------------------------------
Baseline accuracy:      {metrics['acc_baseline']:.4f}
RAPPOR-only accuracy:   {metrics['acc_rappor']:.4f}
RAPPOR + DP-seg:        {metrics['acc_dpseg']:.4f}

Accuracy deltas:
  • RAPPOR - Baseline: {metrics['delta_rappor']:.4f}
  • DPseg  - Baseline: {metrics['delta_dpseg']:.4f}

===========================================
STRUCTURAL LEAKAGE REPORT
-------------------------------------------
Baseline leaf count:      {metrics['leaves_base']}
RAPPOR leaf count:        {metrics['leaves_rappor']}
DP-seg leaf count:        {metrics['leaves_dpseg']}

Max depth:
  Baseline: {metrics['depth_base']}
  RAPPOR:   {metrics['depth_rappor']}
  DP-seg:   {metrics['depth_dpseg']}

Tree shape similarity:
  Baseline <-> RAPPOR: {metrics['sim_rappor']:.4f}
  Baseline <-> DP-seg: {metrics['sim_dpseg']:.4f}

===========================================
INFORMATION LEAKAGE (MUTUAL INFORMATION)
-------------------------------------------
Baseline MI: {metrics.get('mi_baseline', float('nan')):.6f}
RAPPOR MI:   {metrics.get('mi_rappor', float('nan')):.6f}
DP-seg MI:   {metrics.get('mi_dpseg', float('nan')):.6f}

MI ratios (relative to baseline):
  RAPPOR / Baseline: {metrics.get('mi_ratio_rappor', float('nan')):.4f}
  DP-seg / Baseline: {metrics.get('mi_ratio_dpseg', float('nan')):.4f}

===========================================
FILES
-------------------------------------------
baseline_tree.json
rappor_tree.json
dpseg_tree.json
baseline_model.joblib
rappor_model.joblib
dpseg_model.joblib
metadata.json

===========================================
NOTES
-------------------------------------------
Lower similarity scores indicate stronger
structural privacy effects.
Fewer leaves indicate reduced memorization risk.
Lower MI ratios indicate stronger privacy.
    """

    with open(os.path.join(drug_dir, "README.txt"), "w", encoding="utf-8") as f:
        f.write(readme.strip())


def count_leaves(tree_json):
    '''
    Returns number of leaf nodes in tree JSON.
    Smaller trees leak less structure.
    '''
    if tree_json["type"] == "leaf":
        return 1
    return count_leaves(tree_json["left"]) + count_leaves(tree_json["right"])


def tree_depth(tree_json):
    '''
    Returns maximum tree depth.
    '''
    if tree_json["type"] == "leaf":
        return 1
    return 1 + max(tree_depth(tree_json["left"]),
                   tree_depth(tree_json["right"]))


def flatten_tree(tree_json):
    '''
    Converts tree structure into
    linearized representation for comparison.

    Only structure is preserved, not thresholds.
    '''

    structure = []

    def walk(node):
        if node["type"] == "leaf":
            structure.append("L")
            return
        structure.append("N")
        walk(node["left"])
        walk(node["right"])

    walk(tree_json)
    return structure


def jaccard_similarity(a, b):
    '''
    Measures similarity between flattened trees.
    '''
    A = set(enumerate(a))
    B = set(enumerate(b))
    return len(A & B) / max(len(A | B), 1)


def compute_mutual_information(X, y):
    '''
    Computes MI between:
        - feature and label

    Higher MI = higher potential inference risk.
    '''
    return mutual_info_score(X, y)


## **Global Distortion Analysis (RAPPOR vs True Distribution)**

Before evaluating classification performance, we measure how much RAPPOR alone
distorts the global category distribution.

Using the full dataset (without labels), we compute:

- **Jensen–Shannon (JS) divergence**  
  Quantifies how much the decoded distribution differs from the true
  distribution.

- **Exact match rate**  
  Fraction of categories recovered exactly via resampling from the decoded
  distribution.

- **Hamming distance**  
  One minus the match rate, representing structural distortion.

This provides a baseline privacy–utility measure for RAPPOR independent of
model training or segmentation.


In [14]:
##############################################################
# GLOBAL RAPPOR DISTORTION BENCHMARK
#
# Measures how much RAPPOR alone alters the category distribution.
# This is computed ONCE(!!!) using the entire population, independent
# of different tree models.
#
# Purpose:
#   See the privacy–utility tradeoff of RAPPOR alone before
#   DP-seg and anything else are introduced.
##############################################################

print("=======================================")
print("      GLOBAL RAPPOR DISTORTION")
print("=======================================\n")

# Full dataset of valid categories
X_all = X_filtered
K_global = len(mapping_features)

# ---------------------------
#  Distribution
# ---------------------------

true_counts_global = np.bincount(X_all, minlength=K_global)
true_probs_global  = true_counts_global / true_counts_global.sum()

# ---------------------------
# RAPPOR encode + decode
# ---------------------------

reports_global = rappor_client_reports(X_all, seed=999)
counts_global, n_global = rappor_aggregate(reports_global)
est_counts_global, est_probs_global = rappor_decode(counts_global, n_global, K_global)

# i like to be safe 
est_probs_global = np.clip(est_probs_global, 0, None)
est_probs_global = est_probs_global / est_probs_global.sum()

# ---------------------------
# Distribution-level utility
# ---------------------------

js_global = js_divergence(true_probs_global, est_probs_global)

print(f"JS divergence (global): {js_global:.6f}")

# ---------------------------
# Measuring.... 
# ---------------------------

# resample pseudo-categories from decoded distribution
rng = np.random.default_rng(2025)
X_rappor_global = rng.choice(K_global, size=len(X_all), p=est_probs_global)

#(the measuring in question...)
category_match_global = np.mean(X_rappor_global == X_all) # How often does RAPPOR output the original value?
category_hamming_global = 1 - category_match_global # record changes bit-by-bit --> How many bits are flipped in the RAPPOR encoding

print(f"Category match rate (global): {category_match_global:.4f}")
print(f"Category distance (global): {category_hamming_global:.4f}")

print("\n(Computed once; no need to repeat per-drug.)")


      GLOBAL RAPPOR DISTORTION

JS divergence (global): 0.529801
Category match rate (global): 0.0011
Category distance (global): 0.9989

(Computed once; no need to repeat per-drug.)


## **Full Pipeline Evaluation Across 13 Drug Outcomes**

For each NSDUH drug category, the complete pipeline is executed independently.

Per drug, we compute and record:

- total sample size (`n`)  
- positive-class prevalence  
- baseline accuracy (no privacy)  
- RAPPOR-only accuracy  
- RAPPOR + DP-seg accuracy  
- mapping examples (raw --> RAPPOR --> DP-seg)  
- tree structure statistics (depth, leaf count)  
- tree similarity to baseline  
- mutual information with labels  
- distortion metrics (match rate, Hamming distance)  

All results are saved per drug along with:

- trained models  
- JSON tree structures  
- metadata files  
- experiment summary README  

This produces a detailed benchmarking record of how privacy mechanisms impact
learning behavior across different drug outcomes.


In [15]:
##############################################################
# CLEAN PER-DRUG TREE PIPELINE   ദ്ദി/ᐠ｡‸｡ᐟ\
#
# For each drug:          
#   1) extract labels
#   2) train baseline tree                 
#   3) train RAPPOR-only tree
#   4) apply DP segmentation
#   5) train DP-seg tree
#   6) save models + trees
#   7) compute structural leakage
#   8) log results
##############################################################

results = []
ALL_DRUGS = list(DRUG_VARIABLES.keys())

ROOT_DIR = "saved_trees"
os.makedirs(ROOT_DIR, exist_ok=True)


for drug in ALL_DRUGS:

    print("\n==========================================================")
    print(f"                 DRUG: {drug.upper()}")
    print("==========================================================\n")

    # create output folder
    drug_dir = os.path.join(ROOT_DIR, drug.lower())
    os.makedirs(drug_dir, exist_ok=True)

    #######################################################################
    # 1. LOAD LABELS & FILTER FEATURES
    #######################################################################

    y_raw, mask_y = make_drug_labels(df_filtered, drug)
    X_cat = X_filtered[mask_y]
    y     = y_raw

    n = len(y)
    prevalence = y.mean()

    print(f"n = {n}")
    print(f"Prevalence = {prevalence:.4f}\n")

    #######################################################################
    # 2. BASELINE TREE (RAW CATEGORIES)
    #######################################################################

    acc_baseline, tree_base = train_eval_tree_1d(
        X_cat, y, max_depth=4, random_state=42
    )
    print(f"Baseline tree accuracy:      {acc_baseline:.4f}")

    #######################################################################
    # 3. RAPPOR-ONLY TREE (DISTRIBUTION-LEVEL VIEW)
    #
    # Use the decoded global distribution est_probs_global to
    # simulate privatized feature categories. The server sees:
    #   - the decoded histogram (not individual reports)
    #   - we then model its learning power by sampling synthetic
    #     categories from that histogram.
    #######################################################################

    # vary RNG per drug to avoid identical synthetic streams
    rng = np.random.default_rng(hash(drug) % 10_000)
    X_rappor_user = rng.choice(K_global, size=n, p=est_probs_global)

    acc_rappor, tree_rappor = train_eval_tree_1d(
        X_rappor_user, y, max_depth=4, random_state=42
    )
    print(f"RAPPOR-only tree accuracy:   {acc_rappor:.4f}")

    #######################################################################
    # 4. DP-SEGMENTATION TREE (ON TOP OF RAPPOR)
    #
    # DP-Seg uses:
    #   - decoded RAPPOR distribution (est_probs_global)
    #   - label skew per true category (cat_rate)
    # Then:
    #   - segments categories into B clusters
    #   - maps the RAPPOR-sampled categories into segments
    #   - trains a tree on these coarse segments
    #######################################################################

    # compute per-category label prevalence
    cat_total, cat_pos = category_label_counts(X_cat, y, K_global)
    cat_rate = np.zeros(K_global)

    mask_nonzero = (cat_total > 0)
    cat_rate[mask_nonzero] = cat_pos[mask_nonzero] / cat_total[mask_nonzero]

    # clustering categories via DP segmentation
    B = 30
    alpha = 0.7
    
    segment_id, segments = dp_segment_label_aware(
        est_probs_global, cat_rate, B=B, alpha=alpha
    )

    # map RAPPOR-sampled categories to their segments
    X_seg = segment_id[X_rappor_user]

    acc_dpseg, tree_dpseg = train_eval_tree_1d(
        X_seg, y, max_depth=4, random_state=42
    )
    print(f"RAPPOR + DP-seg accuracy:    {acc_dpseg:.4f}\n")

    #######################################################################
    # 5. SAVE MODELS
    #######################################################################

    dump(tree_base,   os.path.join(drug_dir, "baseline_model.joblib"))
    dump(tree_rappor, os.path.join(drug_dir, "rappor_model.joblib"))
    dump(tree_dpseg,  os.path.join(drug_dir, "dpseg_model.joblib"))

    print(f"[SAVED] Models saved to: {drug_dir}")

    #######################################################################
    # 6. EXPORT TREE STRUCTURE (JSON)
    #######################################################################

    baseline_json = sklearn_tree_to_json(tree_base)
    rappor_json   = sklearn_tree_to_json(tree_rappor)
    dpseg_json    = sklearn_tree_to_json(tree_dpseg)

    with open(os.path.join(drug_dir, "baseline_tree.json"), "w") as f:
        json.dump(baseline_json, f, indent=4)

    with open(os.path.join(drug_dir, "rappor_tree.json"), "w") as f:
        json.dump(rappor_json, f, indent=4)

    with open(os.path.join(drug_dir, "dpseg_tree.json"), "w") as f:
        json.dump(dpseg_json, f, indent=4)

    # experiment metadata
    meta_info = {
        "drug": drug,
        "n": n,
        "prevalence": prevalence,
        "B": B,
        "alpha": alpha,
        "max_depth": 4
    }

    with open(os.path.join(drug_dir, "metadata.json"), "w") as f:
        json.dump(meta_info, f, indent=4)

    print(f"[SAVED] JSON trees + metadata -> {drug_dir}")

    #######################################################################
    # 7. STRUCTURAL LEAKAGE ANALYSIS + README GENERATION
    #######################################################################

    # tree complexity
    leaves_base   = count_leaves(baseline_json)
    leaves_rappor = count_leaves(rappor_json)
    leaves_dpseg  = count_leaves(dpseg_json)

    depth_base   = tree_depth(baseline_json)
    depth_rappor = tree_depth(rappor_json)
    depth_dpseg  = tree_depth(dpseg_json)

    # structural comparison
    struct_base   = flatten_tree(baseline_json)
    struct_rappor = flatten_tree(rappor_json)
    struct_dpseg  = flatten_tree(dpseg_json)

    sim_rappor = jaccard_similarity(struct_base, struct_rappor)
    sim_dpseg  = jaccard_similarity(struct_base, struct_dpseg)
    
    # update metadata.json with similarity so plots work
    # aka, last min add, don't want to compute all this before so
    # attemopt at quick fix :D
    meta_info["sim_baseline_rappor"] = float(sim_rappor)
    meta_info["sim_baseline_dpseg"]  = float(sim_dpseg)

    with open(os.path.join(drug_dir, "metadata.json"), "w") as f:
        json.dump(meta_info, f, indent=4)
    
    # for README gen on each drug
    metrics = {
        "acc_baseline": acc_baseline,      # acc -> (# of correct pred) / (total samples)
        "acc_rappor": acc_rappor,
        "acc_dpseg": acc_dpseg,
        
        "delta_rappor": acc_rappor - acc_baseline, # delta_R -> how much accuracy changed after applying RAPPOR
        "delta_dpseg": acc_dpseg - acc_baseline,   # delta_Dp -> how much accuracy changed after applying DP_seg
        
        "leaves_base": leaves_base,        # num of leaves in tree
        "leaves_rappor": leaves_rappor,
        "leaves_dpseg": leaves_dpseg,
        
        "depth_base": depth_base,         # depth of tree
        "depth_rappor": depth_rappor,
        "depth_dpseg": depth_dpseg,
        
        "sim_rappor": sim_rappor,         # jaccard similarity ( baseline <-> RAPPOR )
        "sim_dpseg": sim_dpseg            # jaccard similarity ( baseline <-> DP_Seg )
    }

    write_experiment_readme(drug_dir, meta_info, metrics)

    #######################################################################
    # 8. DISTORTION METRICS
    #######################################################################

    category_match_rappor  = np.mean(X_rappor_user == X_cat)    # RAPPOR match to baseline?
    category_hamming_rappor = 1 - category_match_rappor         # how dif from baseline?

    seg_true = segment_id[X_cat]   # seg assignment if used real category
    category_match_dpseg   = np.mean(X_seg == seg_true)         # X_seg = segment fron RAPPOR (compares to true)
    category_hamming_dpseg = 1 - category_match_dpseg           # how dif from baseline?

    #######################################################################
    # 9. MUTUAL INFORMATION (FULL-RESOLUTION PIPELINE)
    #######################################################################
    mi_baseline = compute_mutual_information(X_cat, y)
    mi_rappor   = compute_mutual_information(X_rappor_user, y)
    mi_dpseg    = compute_mutual_information(X_seg, y)

    mi_ratio_rappor = mi_rappor / mi_baseline if mi_baseline > 0 else 0.0
    mi_ratio_dpseg  = mi_dpseg  / mi_baseline if mi_baseline > 0 else 0.0


    #######################################################################
    # 10. SAMPLE OUTPUT
    #######################################################################

    idx_sample = np.random.choice(n, size=min(15, n), replace=False)
    print("Sample mapping (Raw --> RAPPOR --> DP-seg)")
    print("Raw:     ", X_cat[idx_sample])
    print("RAPPOR:  ", X_rappor_user[idx_sample])
    print("DP-seg:  ", X_seg[idx_sample])

    #######################################################################
    # 11. LOG RESULTS
    #######################################################################

    results.append({
        "drug": drug,
        "n": n,
        "prevalence": prevalence,
        "B": B,
        "alpha": alpha,
        
        "acc_baseline": acc_baseline,
        "acc_rappor": acc_rappor,
        "acc_dpseg": acc_dpseg,

        "delta_rappor": acc_rappor - acc_baseline,
        "delta_dpseg": acc_dpseg - acc_baseline,

        "match_rappor": category_match_rappor,
        "hamming_rappor": category_hamming_rappor,
        "match_dpseg": category_match_dpseg,
        "hamming_dpseg": category_hamming_dpseg,
        "sim_rappor": sim_rappor,

        "sim_dpseg": sim_dpseg,

        "leaves_base": leaves_base,
        "leaves_rappor": leaves_rappor,
        "leaves_dpseg": leaves_dpseg,

        "depth_base": depth_base,
        "depth_rappor": depth_rappor,
        "depth_dpseg": depth_dpseg,

        "mi_baseline": mi_baseline,
        "mi_rappor": mi_rappor,
        "mi_dpseg": mi_dpseg,
        "mi_ratio_rappor": mi_ratio_rappor,
        "mi_ratio_dpseg": mi_ratio_dpseg
    })


##############################################################
# SUMMARY TABLE !!! (ㅅ´ ˘ `)
##############################################################

results_df = pd.DataFrame(results)

print("\n==================== SUMMARY FOR ALL DRUGS ====================")
print(results_df)

results_df.to_csv("final_tree_rappor_dpseg_results.csv", index=False)
print("\nSaved results -> final_tree_rappor_dpseg_results.csv")



                 DRUG: MARIJUANA

n = 173729
Prevalence = 0.2363

Baseline tree accuracy:      0.7637
RAPPOR-only tree accuracy:   0.7637
RAPPOR + DP-seg accuracy:    0.7637

[SAVED] Models saved to: saved_trees\marijuana
[SAVED] JSON trees + metadata -> saved_trees\marijuana
Sample mapping (Raw --> RAPPOR --> DP-seg)
Raw:      [116 215 251   4 104 235  36  82   7  28   1  77 970 208  56]
RAPPOR:   [ 997  533 1417  836 1218  129   93 1528  540  309  584 1122  132 1075
 1074]
DP-seg:   [ 0  5  0  7 19 11 17 29 10  4  6 29 19 12  0]

                 DRUG: COCAINE

n = 173729
Prevalence = 0.0194

Baseline tree accuracy:      0.9806
RAPPOR-only tree accuracy:   0.9806
RAPPOR + DP-seg accuracy:    0.9806

[SAVED] Models saved to: saved_trees\cocaine
[SAVED] JSON trees + metadata -> saved_trees\cocaine
Sample mapping (Raw --> RAPPOR --> DP-seg)
Raw:      [ 13 179 241   1  57 104  82  82  32  42  31  36  31  12 103]
RAPPOR:   [ 753 1396 1012 1636  914 1184 1028  259  270  576   36 1239  857

In [16]:
##############################################################
# GLOBAL DRUG-GROUP TREES (ILLICIT / NICOTINE / PRES-MISUSE)
##############################################################

group_results = []
GROUP_ROOT_DIR = os.path.join(ROOT_DIR, "groups")
os.makedirs(GROUP_ROOT_DIR, exist_ok=True)

for group_name, members in DRUG_GROUPS.items():

    print("\n==========================================================")
    print(f"            DRUG GROUP: {group_name.upper()}")
    print("==========================================================\n")

    # output folder for this group
    group_dir = os.path.join(GROUP_ROOT_DIR, f"group_{group_name}")
    os.makedirs(group_dir, exist_ok=True)

    ###################################################################
    # 1. GROUP LABELS & FEATURE FILTERING
    ###################################################################
    y_group, mask_group = make_group_labels(df_filtered, group_name, members)
    X_cat = X_filtered[mask_group]
    y     = y_group

    n = len(y)
    prevalence = y.mean()

    print(f"n = {n}")
    print(f"Prevalence = {prevalence:.4f}\n")

    ###################################################################
    # 2. BASELINE TREE (RAW CATEGORIES)
    ###################################################################
    acc_baseline, tree_base = train_eval_tree_1d(
        X_cat, y, max_depth=4, random_state=42
    )
    print(f"Baseline tree accuracy:      {acc_baseline:.4f}")

    ###################################################################
    # 3. RAPPOR-ONLY TREE (GLOBAL DECODED DISTRIBUTION)
    ###################################################################
    rng = np.random.default_rng(hash(group_name) % 10_000)
    X_rappor_user = rng.choice(K_global, size=n, p=est_probs_global)

    acc_rappor, tree_rappor = train_eval_tree_1d(
        X_rappor_user, y, max_depth=4, random_state=42
    )
    print(f"RAPPOR-only tree accuracy:   {acc_rappor:.4f}")

    ###################################################################
    # 4. LABEL-AWARE DP SEGMENTATION + DP-SEG TREE
    ###################################################################
    cat_total, cat_pos = category_label_counts(X_cat, y, K_global)
    cat_rate = np.zeros(K_global)
    mask_nonzero = (cat_total > 0)
    cat_rate[mask_nonzero] = cat_pos[mask_nonzero] / cat_total[mask_nonzero]

    B = 30
    alpha = 0.7

    segment_id, segments = dp_segment_label_aware(
        est_probs_global, cat_rate, B=B, alpha=alpha
    )

    X_seg = segment_id[X_rappor_user]

    acc_dpseg, tree_dpseg = train_eval_tree_1d(
        X_seg, y, max_depth=4, random_state=42
    )
    print(f"RAPPOR + DP-seg accuracy:    {acc_dpseg:.4f}\n")

    ###################################################################
    # 5. SAVE MODELS
    ###################################################################
    dump(tree_base,   os.path.join(group_dir, "baseline_model.joblib"))
    dump(tree_rappor, os.path.join(group_dir, "rappor_model.joblib"))
    dump(tree_dpseg,  os.path.join(group_dir, "dpseg_model.joblib"))

    print(f"[SAVED] Group models saved to: {group_dir}")

    ###################################################################
    # 6. EXPORT TREE STRUCTURE (JSON)
    ###################################################################
    baseline_json = sklearn_tree_to_json(tree_base)
    rappor_json   = sklearn_tree_to_json(tree_rappor)
    dpseg_json    = sklearn_tree_to_json(tree_dpseg)

    with open(os.path.join(group_dir, "baseline_tree.json"), "w") as f:
        json.dump(baseline_json, f, indent=4)
    with open(os.path.join(group_dir, "rappor_tree.json"), "w") as f:
        json.dump(rappor_json, f, indent=4)
    with open(os.path.join(group_dir, "dpseg_tree.json"), "w") as f:
        json.dump(dpseg_json, f, indent=4)

    # experiment metadata
    meta_info = {
        "group_name": group_name,
        "members": members,
        "n": n,
        "prevalence": prevalence,
        "B": B,
        "alpha": alpha,
        "max_depth": 4
    }

    with open(os.path.join(group_dir, "metadata.json"), "w") as f:
        json.dump(meta_info, f, indent=4)

    ###################################################################
    # 7. STRUCTURAL LEAKAGE + MI (same metrics as per-drug)
    ###################################################################
    leaves_base   = count_leaves(baseline_json)
    leaves_rappor = count_leaves(rappor_json)
    leaves_dpseg  = count_leaves(dpseg_json)

    depth_base   = tree_depth(baseline_json)
    depth_rappor = tree_depth(rappor_json)
    depth_dpseg  = tree_depth(dpseg_json)

    struct_base   = flatten_tree(baseline_json)
    struct_rappor = flatten_tree(rappor_json)
    struct_dpseg  = flatten_tree(dpseg_json)

    sim_rappor = jaccard_similarity(struct_base, struct_rappor)
    sim_dpseg  = jaccard_similarity(struct_base, struct_dpseg)

    # Mutual information at full resolution
    mi_baseline = compute_mutual_information(X_cat, y)
    mi_rappor   = compute_mutual_information(X_rappor_user, y)
    mi_dpseg    = compute_mutual_information(X_seg, y)

    mi_ratio_rappor = mi_rappor / mi_baseline if mi_baseline > 0 else 0.0
    mi_ratio_dpseg  = mi_dpseg  / mi_baseline if mi_baseline > 0 else 0.0

    # Update metadata and write README for the group
    meta_info["sim_baseline_rappor"] = float(sim_rappor)
    meta_info["sim_baseline_dpseg"]  = float(sim_dpseg)

    with open(os.path.join(group_dir, "metadata.json"), "w") as f:
        json.dump(meta_info, f, indent=4)

    metrics = {
        "acc_baseline": acc_baseline,
        "acc_rappor": acc_rappor,
        "acc_dpseg": acc_dpseg,
        "delta_rappor": acc_rappor - acc_baseline,
        "delta_dpseg": acc_dpseg - acc_baseline,
        "leaves_base": leaves_base,
        "leaves_rappor": leaves_rappor,
        "leaves_dpseg": leaves_dpseg,
        "depth_base": depth_base,
        "depth_rappor": depth_rappor,
        "depth_dpseg": depth_dpseg,
        "sim_rappor": sim_rappor,
        "sim_dpseg": sim_dpseg,
        "mi_baseline": mi_baseline,
        "mi_rappor": mi_rappor,
        "mi_dpseg": mi_dpseg,
        "mi_ratio_rappor": mi_ratio_rappor,
        "mi_ratio_dpseg": mi_ratio_dpseg
    }
    
    write_experiment_readme(group_dir, meta_info, metrics)

    ###################################################################
    # 8. DISTORTION METRICS + LOGGING
    ###################################################################
    category_match_rappor  = np.mean(X_rappor_user == X_cat)
    category_hamming_rappor = 1 - category_match_rappor

    seg_true = segment_id[X_cat]
    category_match_dpseg   = np.mean(X_seg == seg_true)
    category_hamming_dpseg = 1 - category_match_dpseg

    group_results.append({
        "group": group_name,
        "members": ",".join(members),
        "n": n,
        "prevalence": prevalence,
        "B": B,
        "alpha": alpha,
        "acc_baseline": acc_baseline,
        "acc_rappor": acc_rappor,
        "acc_dpseg": acc_dpseg,
        "delta_rappor": acc_rappor - acc_baseline,
        "delta_dpseg": acc_dpseg - acc_baseline,
        "match_rappor": category_match_rappor,
        "hamming_rappor": category_hamming_rappor,
        "match_dpseg": category_match_dpseg,
        "hamming_dpseg": category_hamming_dpseg,
        "sim_rappor": sim_rappor,
        "sim_dpseg": sim_dpseg,
        "leaves_base": leaves_base,
        "leaves_rappor": leaves_rappor,
        "leaves_dpseg": leaves_dpseg,
        "depth_base": depth_base,
        "depth_rappor": depth_rappor,
        "depth_dpseg": depth_dpseg,
        "mi_baseline": mi_baseline,
        "mi_rappor": mi_rappor,
        "mi_dpseg": mi_dpseg,
        "mi_ratio_rappor": mi_ratio_rappor,
        "mi_ratio_dpseg": mi_ratio_dpseg
    })

# Save summary table for groups
group_results_df = pd.DataFrame(group_results)
group_results_df.to_csv(
    os.path.join(GROUP_ROOT_DIR, "global_group_results.csv"),
    index=False
)

print("\n==================== GROUP SUMMARY ====================")
print(group_results_df)
print("\n[SAVED] Global group results ->", os.path.join(GROUP_ROOT_DIR, "global_group_results.csv"))



            DRUG GROUP: ILLICIT

n = 173729
Prevalence = 0.2487

Baseline tree accuracy:      0.7513
RAPPOR-only tree accuracy:   0.7513
RAPPOR + DP-seg accuracy:    0.7513

[SAVED] Group models saved to: saved_trees\groups\group_illicit

            DRUG GROUP: NICOTINE

n = 115739
Prevalence = 0.2596

Baseline tree accuracy:      0.7406
RAPPOR-only tree accuracy:   0.7404
RAPPOR + DP-seg accuracy:    0.7404

[SAVED] Group models saved to: saved_trees\groups\group_nicotine

            DRUG GROUP: PRESCRIPTION_MISUSE

n = 173729
Prevalence = 0.3195

Baseline tree accuracy:      0.6805
RAPPOR-only tree accuracy:   0.6805
RAPPOR + DP-seg accuracy:    0.6805

[SAVED] Group models saved to: saved_trees\groups\group_prescription_misuse

==================== GROUP SUMMARY ====================
                 group                                            members  \
0              illicit  marijuana,cocaine,crack,heroin,methamphetamine...   
1             nicotine                        

## **Resolution Sweep & Privacy–Utility Optimization**

We perform a resolution sweep across multiple domain sizes  
(e.g., K = 1500 -> 30) to study how category resolution affects:

> To clarify, resolution here means how many distinct categories are exposed to the learning algorithm, the domain cardinality. Which is controlled by `K` -> the number of unqiue category values allowed in the feature domain. So, for example, K = 1638 is high resolution, K = 30 is low resolution.

- classification accuracy  
- distribution distortion (JS divergence)  
- information leakage (mutual information)  
- overall privacy–utility tradeoff  

### Category Compression Method

To control resolution, we apply **deterministic index-based compression** to the categorical domain.  
Each original category ID is mapped into one of `K` buckets using uniform linear scaling:

> original_id -> ⌊ (original_id × K) / K_global ⌋

This enforces **representational coarsening** without semantic grouping.

Important clarification:

-  Categories are compressed deterministically  
-  All drugs use the same mapping rule per resolution  
-  Compression is applied before RAPPOR and DP segmentation  
-  Categories are *not* clustered by similarity  
-  No frequency-based merging  
-  No entropy or label-aware binning  

This allows us to isolate the downstream effect of **resolution alone**, independent of domain-specific semantics.

###  Metrics Collected per Resolution

For each drug and resolution, we compute:

- baseline accuracy  
- RAPPOR accuracy  
- DP-Seg accuracy  
- Jensen–Shannon divergence  
- mutual information for each baseline, RAPPOR, and DP-Seg 
- composite privacy–utility score  

###  Best-K Selection Criterion

We identify:

- **best K per drug** (seperated)  
- **best K per drug overall** (combined) 

using a score that balances utility and distortion:
> privacy–utility score = accuracy / (1 + JS divergence)

All results are exported to structured CSV files for analysis and visualization.

In [17]:
##############################################################
# RESOLUTION SWEEP SETUP
#
# Evaluates how category resolution affects:
#   - accuracy
#   - privacy distortion
#   - mutual information
#   - overall privacy–utility tradeoff
##############################################################

# Resolutions to evaluate (big -> small k's)
ALL_K_NEW = [1500, 1000, 500, 200, 100, 50, 30] # Uniform index compression of categorical IDs.

# Output directories
SWEEP_ROOT   = "resolution_sweep"
PER_DRUG_DIR = os.path.join(SWEEP_ROOT, "per_drug")
OVERALL_DIR  = os.path.join(SWEEP_ROOT, "overall")

os.makedirs(PER_DRUG_DIR, exist_ok=True)
os.makedirs(OVERALL_DIR, exist_ok=True)

# Global accumulator
results_all_drugs = []


In [18]:
##############################################################
# FINAL RESOLUTION SWEEP ACROSS ALL DRUGS
#
# For each drug and each resolution K_new:
#   1) compress categories
#   2) baseline model
#   3) RAPPOR encode/decode
#   4) DP segmentation
#   5) evaluate accuracy and leakage
##############################################################

K_global = len(mapping_features) # our og setup, what we got from make_features -> 1638

def compress_categories_by_id(X_cat, K_new, K_global):
    '''
    Reduces category cardinality by integer binning.

    mapping[c] = floor(c * K_new / K_global)

    Parameters:
        X_cat (array-like):
            Original category IDs.
        K_new (int):
            Target resolution.
        K_global (int):
            Original cardinality.

    Return value(s):
        X_comp (np.array):
            Compressed category IDs.
        K_new_eff (int):
            Effective number of bins.
    '''

    K_new_eff = min(K_new, K_global)
    mapping = (np.arange(K_global) * K_new_eff) // K_global
    return mapping[X_cat], K_new_eff


print("=======================================================")
print("     RUNNING FINAL RESOLUTION SWEEP FOR ALL DRUGS")
print("=======================================================\n")

for drug in DRUG_VARIABLES.keys():

    print(f"\n=== RESOLUTION SWEEP FOR DRUG: {drug.upper()} ===")

    # Loadin the labels
    y_raw, mask_y = make_drug_labels(df_filtered, drug)
    X_cat = X_filtered[mask_y]
    y     = y_raw

    n = len(y)
    prevalence = y.mean()

    print(f"n = {n}, prevalence = {prevalence:.4f}")

    per_drug_rows = []

    for K_new in ALL_K_NEW:

        ####################################################
        # 1) Compress domain 
        ####################################################

        X_comp, K_eff = compress_categories_by_id(X_cat, K_new, K_global)

        ####################################################
        # 2) Baseline
        ####################################################

        acc_baseline, _ = train_eval_tree_1d(
            X_comp, y, max_depth=4, random_state=42
        )

        ####################################################
        # 3) True distribution
        ####################################################

        true_counts = np.bincount(X_comp, minlength=K_eff)
        true_probs  = true_counts / true_counts.sum()

        ####################################################
        # 4) RAPPOR encode/decode
        ####################################################

        reports = rappor_client_reports(X_comp, seed=123)
        counts, n_rep = rappor_aggregate(reports)
        est_counts, est_probs = rappor_decode(counts, n_rep, K_eff)

        est_probs = np.clip(est_probs, 0, None)
        est_probs = est_probs / est_probs.sum() if est_probs.sum() > 0 else np.ones(K_eff) / K_eff

        ####################################################
        # 5) Distribution distortion
        ####################################################

        js = js_divergence(true_probs, est_probs)

        ####################################################
        # 6) RAPPOR-only learning
        ####################################################
        
        # vary RNG per resolution to prevent correlated results
        rng = np.random.default_rng(42 + K_eff)
        X_rappor_cat = rng.choice(K_eff, size=n, p=est_probs)

        acc_rappor, _ = train_eval_tree_1d(
            X_rappor_cat, y, max_depth=4, random_state=42
        )

        ####################################################
        # 7) Label-aware DP segmentation
        ####################################################

        cat_total, cat_pos = category_label_counts(X_comp, y, K_eff)
        cat_rate = np.zeros(K_eff)

        mask_nonzero = (cat_total > 0)
        cat_rate[mask_nonzero] = cat_pos[mask_nonzero] / cat_total[mask_nonzero]

        B = min(30, K_eff)
        alpha = 0.7

        segment_id, segments = dp_segment_label_aware(
            est_probs, cat_rate, B=B, alpha=alpha
        )

        X_seg = segment_id[X_rappor_cat]


        ####################################################
        # 8) DP-seg tree
        ####################################################

        acc_dpseg, _ = train_eval_tree_1d(
            X_seg, y, max_depth=4, random_state=42
        )

        ####################################################
        # 9) Mutual information (inference risk proxy)
        ####################################################

        mi_baseline = compute_mutual_information(X_comp, y)
        mi_rappor   = compute_mutual_information(X_rappor_cat, y)
        mi_dpseg    = compute_mutual_information(X_seg, y)

        # Fraction of og info an attacker can still see after RAPPOR
        mi_ratio_rappor = mi_rappor / mi_baseline if mi_baseline > 0 else 0 
        # Fraction of og info an attacker can still see after DP_Seg
        mi_ratio_dpseg  = mi_dpseg  / mi_baseline if mi_baseline > 0 else 0

        ####################################################
        # 10) Composite score
        ####################################################

        privacy_utility_score = acc_dpseg / (1 + js)

        ####################################################
        # 11) Log results
        ####################################################

        row = {
            "drug": drug,
            "K_new": K_eff,
            "B": B,
            "alpha": alpha,
            
            "acc_baseline": acc_baseline,
            "acc_rappor": acc_rappor,
            "acc_dpseg": acc_dpseg,
            
            "mi_baseline": mi_baseline,
            "mi_rappor": mi_rappor,
            "mi_dpseg": mi_dpseg,
            "mi_ratio_rappor": mi_ratio_rappor,
            "mi_ratio_dpseg": mi_ratio_dpseg,
            
            "privacy_utility_score": privacy_utility_score,
            "js_divergence": js,
            "n": n,
            "prevalence": prevalence
        }

        results_all_drugs.append(row)
        per_drug_rows.append(row)

        # persist per-drug progress
        drug_dir = os.path.join(PER_DRUG_DIR, drug.lower())
        os.makedirs(drug_dir, exist_ok=True)

        pd.DataFrame(per_drug_rows).to_csv(
            os.path.join(drug_dir, "resolution_results.csv"),
            index=False
        )

        print(f"[SAVED] Results @ K={K_eff} -> {drug_dir}")

##############################################################
# AGGREGATE SUMMARY
##############################################################

df_all = pd.DataFrame(results_all_drugs)

print("\n================ FINAL RESOLUTION SWEEP RESULTS ================")
print(df_all)

df_all.to_csv(os.path.join(OVERALL_DIR, "all_drugs_resolution.csv"), index=False)
print(f"[SAVED] Aggregate sweep -> {OVERALL_DIR}")

##############################################################
# BEST-K SELECTION (GLOBAL & PER DRUG)
##############################################################

best_overall = df_all.loc[df_all["privacy_utility_score"].idxmax()]

print("\n========== BEST K OVERALL ==========")
print(best_overall)

best_per_drug = df_all.loc[df_all.groupby("drug")["privacy_utility_score"].idxmax()]

print("\n========== BEST K PER DRUG ==========")
print(best_per_drug[["drug", "K_new", "acc_dpseg", "js_divergence", "mi_dpseg", "privacy_utility_score"]])

best_per_drug.to_csv(os.path.join(OVERALL_DIR, "best_k_per_drug_summary.csv"), index=False)

##############################################################
# BEST K BY MUTUAL INFORMATION 
##############################################################

best_k_by_mi = df_all.loc[df_all.groupby("drug")["mi_dpseg"].idxmax()]
best_k_by_mi.to_csv(os.path.join(OVERALL_DIR, "best_k_per_drug_by_mi.csv"), index=False)

print("[SAVED] Best K by MI ->", os.path.join(OVERALL_DIR, "best_k_per_drug_by_mi.csv"))

     RUNNING FINAL RESOLUTION SWEEP FOR ALL DRUGS


=== RESOLUTION SWEEP FOR DRUG: MARIJUANA ===
n = 173729, prevalence = 0.2363
[SAVED] Results @ K=1500 -> resolution_sweep\per_drug\marijuana
[SAVED] Results @ K=1000 -> resolution_sweep\per_drug\marijuana
[SAVED] Results @ K=500 -> resolution_sweep\per_drug\marijuana
[SAVED] Results @ K=200 -> resolution_sweep\per_drug\marijuana
[SAVED] Results @ K=100 -> resolution_sweep\per_drug\marijuana
[SAVED] Results @ K=50 -> resolution_sweep\per_drug\marijuana
[SAVED] Results @ K=30 -> resolution_sweep\per_drug\marijuana

=== RESOLUTION SWEEP FOR DRUG: COCAINE ===
n = 173729, prevalence = 0.0194
[SAVED] Results @ K=1500 -> resolution_sweep\per_drug\cocaine
[SAVED] Results @ K=1000 -> resolution_sweep\per_drug\cocaine
[SAVED] Results @ K=500 -> resolution_sweep\per_drug\cocaine
[SAVED] Results @ K=200 -> resolution_sweep\per_drug\cocaine
[SAVED] Results @ K=100 -> resolution_sweep\per_drug\cocaine
[SAVED] Results @ K=50 -> resolution_sweep\per_

In [23]:
##############################################################
# RESOLUTION SWEEP FOR GLOBAL DRUG GROUPS
#
# For each drug GROUP and each resolution K_new:
#   1) compress categories
#   2) baseline model
#   3) RAPPOR encode/decode
#   4) DP segmentation
#   5) evaluate accuracy and leakage
##############################################################

GROUP_SWEEP_ROOT   = os.path.join(SWEEP_ROOT, "groups")
GROUP_OVERALL_DIR  = os.path.join(GROUP_SWEEP_ROOT, "overall")
os.makedirs(GROUP_SWEEP_ROOT, exist_ok=True)
os.makedirs(GROUP_OVERALL_DIR, exist_ok=True)

results_all_groups = []

print("\n=======================================================")
print("   RUNNING RESOLUTION SWEEP FOR GLOBAL DRUG GROUPS")
print("=======================================================\n")

for group_name, members in DRUG_GROUPS.items():

    print(f"\n=== RESOLUTION SWEEP FOR GROUP: {group_name.upper()} ===")
    print(f"Members: {', '.join(members)}")

    # Build group labels and aligned features
    y_group, mask_group = make_group_labels(df_filtered, group_name, members)
    X_cat = X_filtered[mask_group]
    y     = y_group

    n = len(y)
    prevalence = y.mean()
    print(f"n = {n}, prevalence = {prevalence:.4f}")

    per_group_rows = []

    # Directory for this group's resolution results
    group_dir = os.path.join(GROUP_SWEEP_ROOT, f"group_{group_name}")
    os.makedirs(group_dir, exist_ok=True)

    for K_new in ALL_K_NEW:

        ####################################################
        # 1) Compress domain
        ####################################################
        X_comp, K_eff = compress_categories_by_id(X_cat, K_new, K_global)

        ####################################################
        # 2) Baseline (compressed categories, no DP)
        ####################################################
        acc_baseline, _ = train_eval_tree_1d(
            X_comp, y, max_depth=4, random_state=42
        )

        ####################################################
        # 3) True distribution (compressed)
        ####################################################
        true_counts = np.bincount(X_comp, minlength=K_eff)
        true_probs  = true_counts / true_counts.sum()

        ####################################################
        # 4) RAPPOR encode/decode
        ####################################################
        reports = rappor_client_reports(X_comp, seed=123)
        counts, n_rep = rappor_aggregate(reports)
        est_counts, est_probs = rappor_decode(counts, n_rep, K_eff)

        est_probs = np.clip(est_probs, 0, None)
        est_probs = est_probs / est_probs.sum() if est_probs.sum() > 0 else np.ones(K_eff) / K_eff

        ####################################################
        # 5) Distribution distortion
        ####################################################
        js = js_divergence(true_probs, est_probs)

        ####################################################
        # 6) RAPPOR-only learning
        ####################################################
        rng = np.random.default_rng(42 + K_eff)
        X_rappor_cat = rng.choice(K_eff, size=n, p=est_probs)

        acc_rappor, _ = train_eval_tree_1d(
            X_rappor_cat, y, max_depth=4, random_state=42
        )

        ####################################################
        # 7) Label-aware DP segmentation
        ####################################################
        cat_total, cat_pos = category_label_counts(X_comp, y, K_eff)
        cat_rate = np.zeros(K_eff)
        mask_nonzero = (cat_total > 0)
        cat_rate[mask_nonzero] = cat_pos[mask_nonzero] / cat_total[mask_nonzero]

        B = min(30, K_eff)
        alpha = 0.7

        segment_id, segments = dp_segment_label_aware(
            est_probs, cat_rate, B=B, alpha=alpha
        )

        X_seg = segment_id[X_rappor_cat]

        ####################################################
        # 8) DP-seg tree
        ####################################################
        acc_dpseg, _ = train_eval_tree_1d(
            X_seg, y, max_depth=4, random_state=42
        )

        ####################################################
        # 9) Mutual information (inference risk proxy)
        ####################################################
        mi_baseline = compute_mutual_information(X_comp, y)
        mi_rappor   = compute_mutual_information(X_rappor_cat, y)
        mi_dpseg    = compute_mutual_information(X_seg, y)

        mi_ratio_rappor = mi_rappor / mi_baseline if mi_baseline > 0 else 0.0
        mi_ratio_dpseg  = mi_dpseg  / mi_baseline if mi_baseline > 0 else 0.0

        ####################################################
        # 10) Composite privacy–utility score
        ####################################################
        privacy_utility_score = acc_dpseg / (1 + js)

        ####################################################
        # 11) Log results
        ####################################################
        row = {
            "group": group_name,
            "members": ",".join(members),
            "K_new": K_eff,
            "B": B,
            "alpha": alpha,
            "acc_baseline": acc_baseline,
            "acc_rappor": acc_rappor,
            "acc_dpseg": acc_dpseg,
            "mi_baseline": mi_baseline,
            "mi_rappor": mi_rappor,
            "mi_dpseg": mi_dpseg,
            "mi_ratio_rappor": mi_ratio_rappor,
            "mi_ratio_dpseg": mi_ratio_dpseg,
            "privacy_utility_score": privacy_utility_score,
            "js_divergence": js,
            "n": n,
            "prevalence": prevalence
        }

        results_all_groups.append(row)
        per_group_rows.append(row)

        # Persist per-group progress
        pd.DataFrame(per_group_rows).to_csv(
            os.path.join(group_dir, "resolution_results.csv"),
            index=False
        )

        print(f"[SAVED] Group {group_name} @ K={K_eff} -> {group_dir}")

##############################################################
# AGGREGATE SUMMARY FOR GROUPS
##############################################################

df_groups = pd.DataFrame(results_all_groups)

print("\n============ GROUP RESOLUTION SWEEP RESULTS ============")
print(df_groups)

df_groups.to_csv(
    os.path.join(GROUP_OVERALL_DIR, "all_groups_resolution.csv"),
    index=False
)
print(f"[SAVED] Group aggregate sweep -> {GROUP_OVERALL_DIR}")

##############################################################
# BEST-K SELECTION (GLOBAL & PER GROUP)
##############################################################

best_overall_group = df_groups.loc[df_groups["privacy_utility_score"].idxmax()]

print("\n========== BEST K OVERALL (GROUPS) ==========")
print(best_overall_group)

best_per_group = df_groups.loc[
    df_groups.groupby("group")["privacy_utility_score"].idxmax()
]

print("\n========== BEST K PER GROUP ==========")
print(best_per_group[["group", "K_new", "acc_dpseg", "js_divergence", "mi_dpseg", "privacy_utility_score"]])

best_per_group.to_csv(
    os.path.join(GROUP_OVERALL_DIR, "best_k_per_group_summary.csv"),
    index=False
)

##############################################################
# BEST K BY MUTUAL INFORMATION (GROUPS)
##############################################################

best_k_by_mi_groups = df_groups.loc[
    df_groups.groupby("group")["mi_dpseg"].idxmax()
]
best_k_by_mi_groups.to_csv(
    os.path.join(GROUP_OVERALL_DIR, "best_k_per_group_by_mi.csv"),
    index=False
)

print("[SAVED] Best K by MI (groups) ->", os.path.join(GROUP_OVERALL_DIR, "best_k_per_group_by_mi.csv"))



   RUNNING RESOLUTION SWEEP FOR GLOBAL DRUG GROUPS


=== RESOLUTION SWEEP FOR GROUP: ILLICIT ===
Members: marijuana, cocaine, crack, heroin, methamphetamine, inhalants, hallucinogens
n = 173729, prevalence = 0.2487
[SAVED] Group illicit @ K=1500 -> resolution_sweep\groups\group_illicit
[SAVED] Group illicit @ K=1000 -> resolution_sweep\groups\group_illicit
[SAVED] Group illicit @ K=500 -> resolution_sweep\groups\group_illicit
[SAVED] Group illicit @ K=200 -> resolution_sweep\groups\group_illicit
[SAVED] Group illicit @ K=100 -> resolution_sweep\groups\group_illicit
[SAVED] Group illicit @ K=50 -> resolution_sweep\groups\group_illicit
[SAVED] Group illicit @ K=30 -> resolution_sweep\groups\group_illicit

=== RESOLUTION SWEEP FOR GROUP: NICOTINE ===
Members: tobacco, nicotine_vaping
n = 115739, prevalence = 0.2596
[SAVED] Group nicotine @ K=1500 -> resolution_sweep\groups\group_nicotine
[SAVED] Group nicotine @ K=1000 -> resolution_sweep\groups\group_nicotine
[SAVED] Group nicotine @ K=

## Graph visuals
### VERY and I mean VERY basic graphs, not pretty, just out, out to exist for now

In [19]:
##############################################################
# VISUALIZATION PIPELINE — PER-DRUG + GLOBAL FIGURES
##############################################################

# --------------------------
# DIRECTORY SETUP
# --------------------------

BASE_FIG = Path("figures")
SUMMARY_DIR = BASE_FIG / "summary"

BASE_FIG.mkdir(exist_ok=True)
SUMMARY_DIR.mkdir(exist_ok=True)

def drug_fig_dir(drug):
    d = BASE_FIG / drug.lower()
    d.mkdir(exist_ok=True)
    return d


##############################################################
# PER-DRUG FIGURES
##############################################################

def plot_accuracy_vs_k(df, drug):
    out = drug_fig_dir(drug)
    sub = df[df["drug"] == drug]

    plt.figure()
    plt.plot(sub["K_new"], sub["acc_baseline"], marker='o', label="Baseline")
    plt.plot(sub["K_new"], sub["acc_rappor"], marker='o', label="RAPPOR")
    plt.plot(sub["K_new"], sub["acc_dpseg"], marker='o', label="DP-seg")
    plt.xscale("log")
    plt.xlabel("Resolution (K)")
    plt.ylabel("Accuracy")
    plt.title(f"Accuracy vs K — {drug}")
    plt.grid(True)
    plt.legend()

    plt.savefig(out / "accuracy_vs_k.png", dpi=300)
    plt.close()
    
def plot_mi_vs_k(df, drug):
    out = drug_fig_dir(drug)
    sub = df[df["drug"] == drug]

    plt.figure()
    plt.plot(sub["K_new"], sub["mi_baseline"], marker='o', label="Baseline MI")
    plt.plot(sub["K_new"], sub["mi_rappor"], marker='o', label="RAPPOR MI")
    plt.plot(sub["K_new"], sub["mi_dpseg"], marker='o', label="DP-seg MI")
    plt.xscale("log")
    plt.xlabel("Resolution (K)")
    plt.ylabel("Mutual Information")
    plt.title(f"Mutual Information vs K — {drug}")
    plt.grid(True)
    plt.legend()

    plt.savefig(out / "mi_vs_k.png", dpi=300)
    plt.close()


def plot_js_vs_k(df, drug):
    out = drug_fig_dir(drug)
    sub = df[df["drug"] == drug]

    plt.figure()
    plt.plot(sub["K_new"], sub["js_divergence"], marker='o')
    plt.xscale("log")
    plt.xlabel("Resolution (K)")
    plt.ylabel("JS Divergence")
    plt.title(f"Distribution Distortion (JS) vs K — {drug}")
    plt.grid(True)

    plt.savefig(out / "js_vs_k.png", dpi=300)
    plt.close()


def plot_frontier(df, drug):
    out = drug_fig_dir(drug)
    sub = df[df["drug"] == drug]

    plt.figure()
    plt.scatter(sub["js_divergence"], sub["mi_dpseg"])
    for _, r in sub.iterrows():
        plt.annotate(str(int(r["K_new"])), (r["js_divergence"], r["mi_dpseg"]))
    plt.xlabel("JS Divergence (privacy)")
    plt.ylabel("Mutual Information (utility)")
    plt.title(f"Privacy–Utility Frontier — {drug}")
    plt.grid(True)

    plt.savefig(out / "privacy_frontier.png", dpi=300)
    plt.close()


##############################################################
# RUN ALL PER-DRUG FIGURES
##############################################################

for drug in df_all["drug"].unique():
    print(f"[PLOTTING] {drug}")
    plot_accuracy_vs_k(df_all, drug)
    plot_mi_vs_k(df_all, drug)
    plot_js_vs_k(df_all, drug)
    plot_frontier(df_all, drug)


##############################################################
# GLOBAL SUMMARY FIGURES 
##############################################################

def plot_mean_mi(df):
    mean = df.groupby("K_new")[["mi_baseline","mi_rappor","mi_dpseg"]].mean()

    plt.figure()
    plt.plot(mean.index, mean["mi_baseline"], marker='o', label="Baseline")
    plt.plot(mean.index, mean["mi_rappor"], marker='o', label="RAPPOR")
    plt.plot(mean.index, mean["mi_dpseg"], marker='o', label="DP-seg")
    plt.xscale("log")
    plt.xlabel("Resolution (K)")
    plt.ylabel("Mean Mutual Information")
    plt.title("Mean MI across all drugs")
    plt.grid(True)
    plt.legend()

    plt.savefig(SUMMARY_DIR / "mean_mi_vs_k.png", dpi=300)
    plt.close()


def plot_global_mi_vs_acc(df):
    plt.figure()
    plt.scatter(df["acc_rappor"], df["mi_rappor"], label="RAPPOR")
    plt.scatter(df["acc_dpseg"], df["mi_dpseg"], label="DP-seg")
    plt.xlabel("Accuracy")
    plt.ylabel("Mutual Information")
    plt.title("Accuracy does NOT measure privacy")
    plt.legend()
    plt.grid(True)

    plt.savefig(SUMMARY_DIR / "mi_vs_accuracy.png", dpi=300)
    plt.close()

##############################################################
# GENERATE SUMMARY PLOTS
##############################################################

print("[PLOTTING] Global summary figures")

plot_mean_mi(df_all)
plot_global_mi_vs_acc(df_all)

print("\n All plots generated and saved under /figures/")    


[PLOTTING] marijuana
[PLOTTING] cocaine
[PLOTTING] crack
[PLOTTING] heroin
[PLOTTING] methamphetamine
[PLOTTING] inhalants
[PLOTTING] hallucinogens
[PLOTTING] tobacco
[PLOTTING] nicotine_vaping
[PLOTTING] opioid_misuse
[PLOTTING] tranquilizer_misuse
[PLOTTING] sedative_misuse
[PLOTTING] stimulant_misuse
[PLOTTING] pain_reliever_misuse
[PLOTTING] Global summary figures

 All plots generated and saved under /figures/


In [20]:
##############################################################
# MATPLOTLIB TREE VISUALIZATION 
##############################################################

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

def save_combined_tree_figure(drug, feature_name="Category"):
    drug_dir = Path("saved_trees") / drug.lower()
    fig_dir = FIG_DIR / drug.lower()
    fig_dir.mkdir(parents=True, exist_ok=True)

    base = joblib.load(drug_dir / "baseline_model.joblib")
    rap  = joblib.load(drug_dir / "rappor_model.joblib")
    dp   = joblib.load(drug_dir / "dpseg_model.joblib")

    fig, axes = plt.subplots(1, 3, figsize=(24, 9))

    tree.plot_tree(
        base, ax=axes[0], filled=True, fontsize=8,
        feature_names=[feature_name], class_names=["No", "Yes"]
    )
    axes[0].set_title("Baseline", fontsize=13)

    tree.plot_tree(
        rap, ax=axes[1], filled=True, fontsize=8,
        feature_names=[feature_name], class_names=["No", "Yes"]
    )
    axes[1].set_title("RAPPOR", fontsize=13)

    tree.plot_tree(
        dp, ax=axes[2], filled=True, fontsize=8,
        feature_names=[feature_name], class_names=["No", "Yes"]
    )
    axes[2].set_title("DP-Seg", fontsize=13)

    fig.suptitle(drug.upper(), fontsize=16)
    plt.tight_layout()
    out = fig_dir / "comparison_tree.png"
    plt.savefig(out, dpi=200)
    plt.close()
    #print(f"[FIGURE SAVED] {out}")

    
    
##############################################################
# Heat Map (show similarity)
##############################################################
SIM_DIR = Path("figures") / "structure_similarity"
SIM_DIR.mkdir(parents=True, exist_ok=True)

def plot_tree_similarity_from_metrics(drug, metrics):
    labels = ["Baseline", "RAPPOR", "DP-Seg"]

    sim_base_rappor = float(metrics["sim_baseline_rappor"])
    sim_base_dpseg  = float(metrics["sim_baseline_dpseg"])

    # If you do not compute RAPPOR <-> DPseg directly, estimate it
    if "sim_rappor_dpseg" in metrics:
        sim_rappor_dpseg = float(metrics["sim_rappor_dpseg"])
    else:
        sim_rappor_dpseg = (sim_base_rappor + sim_base_dpseg) / 2

    sim = np.array([
        [1.0,               sim_base_rappor, sim_base_dpseg],
        [sim_base_rappor,   1.0,             sim_rappor_dpseg],
        [sim_base_dpseg,    sim_rappor_dpseg, 1.0]
    ], dtype=float)

    plt.figure(figsize=(5.5, 4.5))
    im = plt.imshow(sim, vmin=0, vmax=1, cmap="coolwarm")
    plt.colorbar(im)

    plt.xticks(range(3), labels)
    plt.yticks(range(3), labels)

    for i in range(3):
        for j in range(3):
            plt.text(j, i, f"{sim[i,j]:.2f}", ha="center", va="center", color="black")

    plt.title(f"{drug.upper()} — Tree Structural Similarity")
    plt.tight_layout()

    out = SIM_DIR / f"{drug.lower()}_tree_similarity.png"
    plt.savefig(out, dpi=200)
    plt.close()

    #print(f"[SIM FIXED] {out}")

###############
# run em
###############

RESULTS_DIR = Path("saved_trees")

for drug in ALL_DRUGS:
    print(f"[PLOTTING] {drug}")

    # Load per-drug metadata
    meta_path = RESULTS_DIR / drug / "metadata.json"
    
    with open(meta_path, "r") as f:
        metrics = json.load(f)

    # Generate plots
    save_combined_tree_figure(drug)
    plot_tree_similarity_from_metrics(drug, metrics)

[PLOTTING] marijuana
[PLOTTING] cocaine
[PLOTTING] crack
[PLOTTING] heroin
[PLOTTING] methamphetamine
[PLOTTING] inhalants
[PLOTTING] hallucinogens
[PLOTTING] tobacco
[PLOTTING] nicotine_vaping
[PLOTTING] opioid_misuse
[PLOTTING] tranquilizer_misuse
[PLOTTING] sedative_misuse
[PLOTTING] stimulant_misuse
[PLOTTING] pain_reliever_misuse


In [21]:
FIG_DIR = Path("figures/summary")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# table making
df = results_df.copy()
df["sim_rdp"] = (df["sim_rappor"] + df["sim_dpseg"]) / 2   # estimated similarity
df = df.sort_values("drug")  # alphabetical for clean plotting

labels = df["drug"]
baseline_rappor = df["sim_rappor"]
baseline_dpseg  = df["sim_dpseg"]
rappor_dpseg    = df["sim_rdp"]

x = np.arange(len(labels))
width = 0.26

#plot
plt.figure(figsize=(14, 5))

plt.bar(x - width, baseline_rappor, width, label="Baseline <-> RAPPOR")
plt.bar(x,         baseline_dpseg,  width, label="Baseline <-> DP-Seg")
plt.bar(x + width, rappor_dpseg,    width, label="RAPPOR <-> DP-Seg")

plt.xticks(x, labels, rotation=45, ha="right")
plt.ylim(0, 1.05)

plt.ylabel("Structural Similarity (Jaccard Index)")
plt.title("Tree Structural Similarity Across Drugs")

plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)


## gogogo!!!
out = FIG_DIR / "structural_similarity_all_drugs.png"
plt.tight_layout()
plt.savefig(out, dpi=300)
plt.close()

print(f"[SAVED] Combined structural similarity plot -> {out}")


[SAVED] Combined structural similarity plot -> figures\summary\structural_similarity_all_drugs.png


# Tree Structural Similarity — How to Interpret the Figure Above

## High Similarity Values (≈ 1.0)
When similarity is close to 1.0, the two trees are structurally nearly identical.  
This means the models:

- Split on the same features  
- Use similar thresholds  
- Follow the same decision paths  

In short, the models encode the **same internal representation**.

---

## Low Similarity Values (≈ 0.2–0.3)
Low similarity indicates that the trees differ substantially in structure.  
This reflects:

- Different split locations  
- Different feature usage  
- Different logic paths  

The model is **reasoning about the data differently**, even if predictive accuracy appears unchanged.

---

# What the Graphs Show (and values from each drug's individual README)

## CASE 1: RAPPOR Preserves Structure Better Than DP Segmentation
**(Baseline–RAPPOR similarity > Baseline–DP-Seg similarity)**
the drugs in this case would be:

- changed

### Observations

These targets correspond to behaviors that are:
- hmmm


### Interpretation

> 

---


## CASE 2: DP Segmentation Preserves Structure Better Than RAPPOR
**(Baseline–DP-Seg similarity > Baseline–RAPPOR similarity)**
the drugs in this case would be:
 
- changed

### Observations

These drugs exhibit:
- hmmm

hmmmmmmm......
### Interpretation


> 

---

## CASE 3: Both Mechanisms Induce Equal Structural Distortion
**(Baseline–RAPPOR similarity ≈ Baseline–DP-Seg similarity)**
the drugs in this case would be:

- changed...

### Observations

These cases exhibit:
- hmmmm  

hmmmm......?

### Interpretation

> 

---

# Attempt to categorize/organize/cluster

.......

---
## What Is *Not* Driving the Differences (the one thing I am able to see)

There is no evidence that structural behavior depends on:

- 



---

## Summary Conclusion (what I think can be concluded here)

> Privacy mechanisms .. something with that perchance...

This result highlights ..... hm

---

## some takeaways
 that i think can be made....
 
> 

---

Having higher structure preserved values of similarity does result in some....
bad news for privacy strength.... 

| Similarity | Meaning | 
|-----------|--------------|
| 1.0 | Structural privacy failure | 
| < 0.6 | Partial leakage |
| < 0.4 | Strong/Good privacy | 
| < 0.2 | Strict/high privacy |


god im tired